# 08 — Expérimentation Application

Ce notebook sert à **tester et valider** la logique des pages Streamlit avant de l'intégrer dans l'app.

Chaque section correspond à une page ou un composant de l'application :
- **P100 Accueil** — données historiques, classements, graphiques top 6
- **P200 Classement** — saison 2026-27 en cours, probabilités LDC
- **P300 Matchs** — prédictions, features dynamiques
- **P400 Simulation** — Monte Carlo
- **P500 Carte** — stades

**Règle** : rien ne passe dans l'app sans avoir été validé ici d'abord.

In [ ]:
import sys
from pathlib import Path

# Trouver la racine du projet de façon robuste (fonctionne peu importe d'où Jupyter est lancé)
# On cherche le répertoire qui contient à la fois 'data/' et 'app/'
_here = Path().resolve()
for _p in [_here, *_here.parents]:
    if (_p / 'data').exists() and (_p / 'app').exists():
        ROOT = _p
        break

APP = ROOT / 'app'

sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(APP))

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')

print('ROOT :', ROOT)
print('APP  :', APP)
print('data existe :', (ROOT / 'data').exists())


---
## P100 — Accueil (données historiques)

### 1. Chargement des données

In [ ]:
df = pd.read_csv(ROOT / 'data' / 'raw' / 'pl_all_seasons.csv', low_memory=False)

print(f'Lignes       : {len(df):,}')
print(f'Colonnes     : {df.shape[1]}')
print(f'Saisons raw  : {sorted(df["Season"].unique())}')

In [ ]:
print(f'Saisons dans la base : {df["Season"].nunique()}')

In [ ]:
valid_seasons = sorted([s for s in df['Season'].unique() if len(str(int(s))) == 4])
print(f'Saisons valides : {len(valid_seasons)}')
print(valid_seasons)

In [ ]:
for s in sorted(df['Season'].unique()):
    count = len(df[df['Season'] == s])
    print(f"  Season={s}  —  {count} matchs")

print(f"\nTotal : {df['Season'].nunique()} valeurs uniques")


In [ ]:
def fix_season(raw):
    s = str(int(raw)).zfill(4)  # 102 -> "0102", 1 -> "0001"

    y1 = int(s[:2])
    y2 = s[2:]

    if 0 <= y1 <= 9:        # 0001->2000/01, 0102->2001/02 ... 0910->2009/10
        return f"200{s[1]}/{y2}"
    elif 10 <= y1 <= 29:    # 1011->2010/11 ... 2526->2025/26
        return f"20{s[:2]}/{y2}"
    elif y1 >= 93:          # 9394->1993/94 ... 9900->1999/00
        return f"19{s[:2]}/{y2}"
    else:
        return f"UNKNOWN_{s}"

df['Season_display'] = df['Season'].apply(fix_season)

for s in sorted(df['Season'].unique()):
    display = df[df['Season'] == s]['Season_display'].iloc[0]
    count   = len(df[df['Season'] == s])
    print(f"  {s:>6}  ->  {display}  —  {count} matchs")


In [ ]:
# Garder uniquement les saisons 2000/01 -> 2025/26
saisons_cibles = [s for s in df['Season'].unique()
                  if df[df['Season'] == s]['Season_display'].iloc[0].startswith('20')]

df_pl = df[df['Season'].isin(saisons_cibles)].copy().reset_index(drop=True)

# Vérification
print(f"Lignes    : {len(df_pl):,}")
print(f"Saisons   : {df_pl['Season_display'].nunique()}")
print()
for s in sorted(df_pl['Season'].unique()):
    display = df_pl[df_pl['Season'] == s]['Season_display'].iloc[0]
    count   = len(df_pl[df_pl['Season'] == s])
    print(f"  {display}  —  {count} matchs")


In [ ]:
# Parser les deux formats : essai %d/%m/%Y d'abord, puis %d/%m/%y pour le reste
dates = pd.to_datetime(df_pl['Date'], format='%d/%m/%Y', errors='coerce')

mask_failed = dates.isna()
dates[mask_failed] = pd.to_datetime(
    df_pl.loc[mask_failed, 'Date'], format='%d/%m/%y', errors='coerce'
)

df_pl['Date'] = dates

# Vérification
print(f"NaT après parsing  : {df_pl['Date'].isna().sum()}")
print(f"Plage de dates     : {df_pl['Date'].min().date()}  ->  {df_pl['Date'].max().date()}")
print(f"Type               : {df_pl['Date'].dtype}")
print()
# Vérifier par saison que les dates sont cohérentes
for s in sorted(df_pl['Season'].unique()):
    sub = df_pl[df_pl['Season'] == s]
    display = sub['Season_display'].iloc[0]
    print(f"  {display}  :  {sub['Date'].min().date()}  ->  {sub['Date'].max().date()}  |  NaT: {sub['Date'].isna().sum()}")


In [ ]:
# Identifier la ligne avec NaT
nat_row = df_pl[df_pl['Date'].isna()]
print("Ligne avec NaT :")
print(nat_row[['Season', 'Season_display', 'Date', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG', 'FTR']])


In [ ]:
df_pl = df_pl.dropna(subset=['Date', 'HomeTeam', 'AwayTeam']).reset_index(drop=True)

print(f"Lignes après nettoyage : {len(df_pl):,}")
print(f"NaT restants           : {df_pl['Date'].isna().sum()}")
print(f"Saisons                : {df_pl['Season_display'].nunique()}")


### 2. Classement final d'une saison

In [ ]:
def season_standings(df, season_display):
    """
    Classement final d'une saison.
    season_display : '2025/26', '2000/01', etc.
    """
    s = df[df['Season_display'] == season_display].copy()

    if s.empty:
        print(f"Aucune donnée pour la saison {season_display}")
        return pd.DataFrame()

    teams = sorted(set(s['HomeTeam'].dropna()) | set(s['AwayTeam'].dropna()))
    rows  = []

    for team in teams:
        hm = s[s['HomeTeam'] == team]
        am = s[s['AwayTeam'] == team]

        w  = len(hm[hm['FTR'] == 'H']) + len(am[am['FTR'] == 'A'])
        d  = len(hm[hm['FTR'] == 'D']) + len(am[am['FTR'] == 'D'])
        l  = len(hm[hm['FTR'] == 'A']) + len(am[am['FTR'] == 'H'])
        gf = int(hm['FTHG'].sum() + am['FTAG'].sum())
        ga = int(hm['FTAG'].sum() + am['FTHG'].sum())

        rows.append({
            'team'   : team,
            'played' : w + d + l,
            'won'    : w,
            'drawn'  : d,
            'lost'   : l,
            'gf'     : gf,
            'ga'     : ga,
            'gd'     : gf - ga,
            'points' : w * 3 + d,
        })

    out = (pd.DataFrame(rows)
             .sort_values(['points', 'gd', 'gf'], ascending=False)
             .reset_index(drop=True))
    out.insert(0, 'pos', range(1, len(out) + 1))
    return out

# Test
standings = season_standings(df_pl, '2025/26')
print(standings)


### 3. Cinq dernier champion

In [ ]:
def last_5_champions(df, season_display):
    """
    Retourne les 5 derniers champions avant et incluant la saison passée en paramètre.
    season_display : '2025/26', '2010/11', etc.
    """
    FIRST_SEASON = '2000/01'

    available = sorted(df['Season_display'].unique(),
                       key=lambda x: int(x[:4]))

    if season_display not in available:
        print(f"Saison {season_display} non trouvée dans la base.")
        return []

    idx    = available.index(season_display)
    window = available[max(0, idx - 4) : idx + 1]

    if len(window) < 5:
        print(f"Attention : nos données se limitent à {FIRST_SEASON}.")
        print(f"Seules {len(window)} saison(s) disponible(s) sur les 5 demandées.\n")

    results = []
    for season in reversed(window):
        s = season_standings(df, season)
        if s.empty:
            continue
        c = s.iloc[0]
        results.append({
            'saison'  : season,
            'champion': c['team'],
            'points'  : c['points'],
            'won'     : c['won'],
            'gf'      : c['gf'],
            'ga'      : c['ga'],
        })

    return results

# Affichage
def print_champions(results):
    print(f"  {'Saison':<10} {'Champion':<22} {'Pts':>4}  {'V':>3}  {'G/A'}")
    print(f"  {'-'*9} {'-'*21} {'-'*4}  {'-'*3}  {'-'*8}")
    for r in results:
        print(f"  {r['saison']:<10} {r['champion']:<22} {r['points']:>4}  "
              f"{r['won']:>3}V  {r['gf']}/{r['ga']}")

# Test 1
print("=== 2025/26 ===")
print_champions(last_5_champions(df_pl, '2025/26'))

print()

# Test 2
print("=== 2005/06 ===")
print_champions(last_5_champions(df_pl, '2005/06'))


### 3. Évolution des points du top 6 (graphique P100)

In [ ]:
def top6_evolution(df, season_display):
    """
    Points cumulés du top 6 final, journée par journée.
    Retourne un DataFrame : [matchweek, team, cumulative_points]
    """
    s = df[df['Season_display'] == season_display].copy()

    if s.empty:
        print(f"Aucune donnée pour {season_display}")
        return pd.DataFrame()

    # Top 6 final
    final = season_standings(df, season_display)
    top6  = final.head(6)['team'].tolist()

    # Assigner journée via rang des dates
    s = s.dropna(subset=['Date']).sort_values('Date')
    s['matchweek'] = s['Date'].rank(method='dense').astype(int)

    # Calculer les points cumulés
    pts  = {t: 0 for t in top6}
    rows = []

    for mw, grp in s.groupby('matchweek'):
        for _, match in grp.iterrows():
            h, a, r = match['HomeTeam'], match['AwayTeam'], match['FTR']
            if h in pts:
                pts[h] += 3 if r == 'H' else (1 if r == 'D' else 0)
            if a in pts:
                pts[a] += 3 if r == 'A' else (1 if r == 'D' else 0)
        for t in top6:
            rows.append({'matchweek': mw, 'team': t, 'pts': pts[t]})

    evo = pd.DataFrame(rows)
    return evo

# Test
evo = top6_evolution(df_pl, '2025/26')
print(f"Shape     : {evo.shape}")
print(f"NaN mw    : {evo['matchweek'].isna().sum()}")
print(f"Journées  : {evo['matchweek'].nunique()}")
print(f"Équipes   : {evo['team'].unique().tolist()}")
print()
print(evo.tail(6).to_string(index=False))


In [ ]:
def assign_matchweek(df, season_display):
    """
    Assigner le numéro de journée correct (1-38) basé sur
    le rang du match pour chaque équipe.
    Un match = journée N si c'est le N-ième match des deux équipes.
    """
    s = df[df['Season_display'] == season_display].copy()
    s = s.dropna(subset=['Date']).sort_values('Date').reset_index(drop=True)

    # Compteur de matchs joués par équipe
    team_match_count = {}
    matchweeks = []

    for _, row in s.iterrows():
        h = row['HomeTeam']
        a = row['AwayTeam']

        team_match_count[h] = team_match_count.get(h, 0) + 1
        team_match_count[a] = team_match_count.get(a, 0) + 1

        # La journée = max des deux compteurs (les deux équipes doivent avoir joué)
        mw = max(team_match_count[h], team_match_count[a])
        matchweeks.append(mw)

    s['matchweek'] = matchweeks
    return s

# Test
s_test = assign_matchweek(df_pl, '2025/26')
print(f"Journées uniques : {s_test['matchweek'].nunique()}")
print(f"Min / Max        : {s_test['matchweek'].min()} / {s_test['matchweek'].max()}")
print(f"Matchs par journée :")
print(s_test['matchweek'].value_counts().sort_index().to_string())


In [ ]:
def top6_evolution(df, season_display):
    """
    Points cumulés du top 6 final, journée par journée (J1-J38).
    """
    final = season_standings(df, season_display)
    if final.empty:
        return pd.DataFrame()

    top6 = final.head(6)['team'].tolist()

    # Utiliser les journées correctes
    s = assign_matchweek(df, season_display)

    pts  = {t: 0 for t in top6}
    rows = []

    for mw, grp in s.groupby('matchweek'):
        for _, match in grp.iterrows():
            h, a, r = match['HomeTeam'], match['AwayTeam'], match['FTR']
            if h in pts:
                pts[h] += 3 if r == 'H' else (1 if r == 'D' else 0)
            if a in pts:
                pts[a] += 3 if r == 'A' else (1 if r == 'D' else 0)
        for t in top6:
            rows.append({'matchweek': mw, 'team': t, 'pts': pts[t]})

    return pd.DataFrame(rows)

# Test
evo = top6_evolution(df_pl, '2025/26')
print(f"Shape    : {evo.shape}")
print(f"Journées : {evo['matchweek'].nunique()}")
print()
print(evo.tail(6).to_string(index=False))


In [ ]:
COLORS = ['#00C46A', '#2FD9E0', '#FFB100', '#FF3B5C', '#9B59B6', '#E67E22']

def plot_top6(evo, season_display):
    teams = evo['team'].unique().tolist()

    fig = go.Figure()

    for i, team in enumerate(teams):
        td = evo[evo['team'] == team]
        fig.add_trace(go.Scatter(
            x=td['matchweek'],
            y=td['pts'],
            name=team,
            mode='lines',
            line=dict(color=COLORS[i % len(COLORS)], width=2.5),
            hovertemplate=f'<b>{team}</b><br>J%{{x}} — %{{y}} pts<extra></extra>'
        ))

    fig.update_layout(
        title=dict(
            text=f'Fluctuation Top 6 — {season_display}',
            font=dict(size=20, color='#2FD9E0')
        ),
        plot_bgcolor='#0B0E14',
        paper_bgcolor='#12182A',
        font=dict(color='#E8E8E8', family='IBM Plex Mono'),
        xaxis=dict(
            title='Journée (date unique)',
            gridcolor='rgba(139,146,168,0.12)',
            color='#8B92A8',
        ),
        yaxis=dict(
            title='Points cumulés',
            gridcolor='rgba(139,146,168,0.12)',
            color='#8B92A8',
        ),
        hovermode='x unified',
        legend=dict(
            bgcolor='rgba(18,24,42,0.85)',
            bordercolor='#8B92A8',
            borderwidth=1
        ),
        height=460,
        margin=dict(l=10, r=10, t=50, b=10)
    )

    return fig

fig = plot_top6(evo, '2025/26')
fig.show()

### 4. Stats clés

In [ ]:
def season_stats(df, season_display):
    """
    Stats clés d'une saison par équipe.
    """
    s = df[df['Season_display'] == season_display].copy()

    if s.empty:
        print(f"Aucune donnée pour {season_display}")
        return {}

    standings = season_standings(df, season_display)
    if standings.empty:
        return {}

    # Vérifier si les colonnes xG existent
    has_xg = 'Home_xG' in s.columns and 'Away_xG' in s.columns

    # --- Stats par équipe ---
    team_stats = []
    for _, row in standings.iterrows():
        team = row['team']
        hm   = s[s['HomeTeam'] == team]
        am   = s[s['AwayTeam'] == team]
        played = row['played']

        # Buts
        gf = row['gf']
        ga = row['ga']
        gf_per_match = round(gf / played, 2) if played > 0 else 0
        ga_per_match = round(ga / played, 2) if played > 0 else 0

        # Clean sheets
        cs_home = len(hm[hm['FTAG'] == 0])
        cs_away = len(am[am['FTHG'] == 0])
        clean_sheets = cs_home + cs_away

        # Victoires consécutives max (win streak)
        results = []
        for _, m in s[s['HomeTeam'] == team].iterrows():
            results.append((m['Date'], 'W' if m['FTR'] == 'H' else ('D' if m['FTR'] == 'D' else 'L')))
        for _, m in s[s['AwayTeam'] == team].iterrows():
            results.append((m['Date'], 'W' if m['FTR'] == 'A' else ('D' if m['FTR'] == 'D' else 'L')))
        results.sort(key=lambda x: x[0])

        max_streak = cur_streak = 0
        for _, r in results:
            if r == 'W':
                cur_streak += 1
                max_streak  = max(max_streak, cur_streak)
            else:
                cur_streak  = 0

        # xG ratio (si disponible)
        xg_ratio = None
        if has_xg:
            xg_for  = hm['Home_xG'].sum() + am['Away_xG'].sum()
            xg_ratio = round(gf / xg_for, 2) if xg_for > 0 else None

        team_stats.append({
            'team'        : team,
            'played'      : played,
            'points'      : row['points'],
            'gf'          : gf,
            'ga'          : ga,
            'gf_per_match': gf_per_match,
            'ga_per_match': ga_per_match,
            'clean_sheets': clean_sheets,
            'max_streak'  : max_streak,
            'xg_ratio'    : xg_ratio,
        })

    team_df = pd.DataFrame(team_stats)

    # --- Résumé stats clés ---
    best_atk  = team_df.loc[team_df['gf'].idxmax()]
    best_def  = team_df.loc[team_df['ga'].idxmin()]
    best_cs   = team_df.loc[team_df['clean_sheets'].idxmax()]
    best_str  = team_df.loc[team_df['max_streak'].idxmax()]

    stats = {
        'best_attack' : {
            'team'         : best_atk['team'],
            'goals'        : int(best_atk['gf']),
            'per_match'    : best_atk['gf_per_match'],
        },
        'best_defense' : {
            'team'         : best_def['team'],
            'goals'        : int(best_def['ga']),
            'per_match'    : best_def['ga_per_match'],
        },
        'best_clean_sheets' : {
            'team'  : best_cs['team'],
            'count' : int(best_cs['clean_sheets']),
        },
        'best_win_streak' : {
            'team'  : best_str['team'],
            'games' : int(best_str['max_streak']),
        },
    }

    # xG ratio si disponible
    if has_xg:
        best_xg = team_df.dropna(subset=['xg_ratio']).loc[
            team_df.dropna(subset=['xg_ratio'])['xg_ratio'].idxmax()
        ]
        stats['best_xg_ratio'] = {
            'team' : best_xg['team'],
            'ratio': best_xg['xg_ratio'],
        }

    return stats, team_df

# Test
# Test
stats, team_df = season_stats(df_pl, '2025/26')

print("=== Stats clés 2025/26 ===\n")
print(f"Best Attack      : {stats['best_attack']['team']:<20} "
      f"{stats['best_attack']['goals']} buts  "
      f"({stats['best_attack']['per_match']} / match)")

print(f"Best Defense     : {stats['best_defense']['team']:<20} "
      f"{stats['best_defense']['goals']} buts encaissés  "
      f"({stats['best_defense']['per_match']} / match)")

print(f"Best Clean Sheets: {stats['best_clean_sheets']['team']:<20} "
      f"{stats['best_clean_sheets']['count']} clean sheets")

print(f"Best Win Streak  : {stats['best_win_streak']['team']:<20} "
      f"{stats['best_win_streak']['games']} victoires consécutives")

if 'best_xg_ratio' in stats:
    print(f"Best xG Ratio    : {stats['best_xg_ratio']['team']:<20} "
          f"{stats['best_xg_ratio']['ratio']} (buts/xG)")


In [ ]:
import re

england_master = ROOT / 'data' / 'raw' / 'england-master'

# Lister tous les sous-dossiers
folders = sorted([f for f in england_master.iterdir() if f.is_dir()])
print(f"Dossiers disponibles : {len(folders)}")
for f in folders:
    txt = f / '1-premierleague.txt'
    print(f"  {f.name:<12}  {'OK' if txt.exists() else 'MANQUANT'}")


In [ ]:
def parse_top_scorer(season_folder):
    """
    Parse le fichier 1-premierleague.txt et retourne le meilleur buteur.
    """
    txt_file = england_master / season_folder / '1-premierleague.txt'
    if not txt_file.exists():
        return None

    content = txt_file.read_text(encoding='utf-8', errors='ignore')

    # Capturer uniquement les segments qui contiennent des minutes (ex: 45', 90+2')
    # On exclut les scores (1-0), durées (282d), etc.
    goal_pattern = re.compile(r'\(([^)]*\d+\'[^)]*)\)', re.DOTALL)

    scorer_count = {}

    for match in goal_pattern.finditer(content):
        segment = match.group(1)

        # Séparer les deux équipes (séparateur ;)
        for team_part in segment.split(';'):
            # Séparer les buts individuels
            for goal in team_part.split(','):
                goal = goal.strip()
                if not goal:
                    continue

                # Ignorer les contre-son-camp
                if '(og)' in goal:
                    continue

                # Extraire le nom : tout ce qui précède la minute (ex: "87'" ou "90+3'")
                # Pattern minute : espace + chiffres + éventuellement +chiffres + apostrophe
                name = re.sub(r"\s+\d+\+?\d*'.*$", '', goal).strip()

                if name and len(name) > 1:
                    scorer_count[name] = scorer_count.get(name, 0) + 1

    if not scorer_count:
        return None

    sorted_scorers = sorted(scorer_count.items(), key=lambda x: x[1], reverse=True)

    return {
        'name' : sorted_scorers[0][0],
        'goals': sorted_scorers[0][1],
        'all'  : sorted_scorers[:10]
    }

# Test
result = parse_top_scorer('2025-26')
if result:
    print(f"Meilleur buteur : {result['name']}  —  {result['goals']} buts")
    print("\nTop 10 :")
    for name, goals in result['all']:
        print(f"  {name:<35} {goals} buts")


In [ ]:
# Test sur 2024-25
result_2425 = parse_top_scorer('2024-25')
if result_2425:
    print(f"Meilleur buteur 2024/25 : {result_2425['name']}  —  {result_2425['goals']} buts")
    print("\nTop 10 :")
    for name, goals in result_2425['all']:
        print(f"  {name:<35} {goals} buts")

# Diagnostic buts total
txt_file = england_master / '2024-25' / '1-premierleague.txt'
content_2425 = txt_file.read_text(encoding='utf-8', errors='ignore')

scorer_count_2425 = {}
goal_pattern = re.compile(r'\(([^)]*\d+\'[^)]*)\)', re.DOTALL)
for match in goal_pattern.finditer(content_2425):
    segment = match.group(1)
    for team_part in segment.split(';'):
        for goal in team_part.split(','):
            goal = goal.strip()
            if not goal or '(og)' in goal:
                continue
            name = re.sub(r"\s+\d+\+?\d*'.*$", '', goal).strip()
            if name and len(name) > 1:
                scorer_count_2425[name] = scorer_count_2425.get(name, 0) + 1

total = sum(scorer_count_2425.values())
print(f"\nButs parsés total  : {total}")
print(f"Buts attendus ~    : {380 * 2.7:.0f}")

# Lignes tronquées
lines = content_2425.split('\n')
truncated = [l for l in lines if re.search(r'\(p\s*$', l.strip())]
print(f"Lignes avec '(p' tronqué : {len(truncated)}")
for l in truncated[:5]:
    print(f"  {repr(l.strip())}")


In [ ]:
txt_file_2425 = england_master / '2024-25' / '1-premierleague.txt'
content_2425  = txt_file_2425.read_text(encoding='utf-8', errors='ignore')

# Afficher les 60 premières lignes
for i, line in enumerate(content_2425.split('\n')[:60]):
    print(f"{i:>3}  {repr(line)}")


### 5. Validation sur plusieurs saisons

In [ ]:
# Validation sur toutes les saisons disponibles dans df_pl
saisons = sorted(df_pl['Season_display'].unique())

print(f"{'Saison':<12} {'Champion':<22} {'Best Atk':<20} {'Best Def':<20} {'Streak'}")
print("-" * 90)

for s in saisons:
    try:
        stats, _ = season_stats(df_pl, s)
        champ  = season_standings(df_pl, s).iloc[0]['team']
        print(f"{s:<12} {champ:<22} "
              f"{stats['best_attack']['team']:<20} "
              f"{stats['best_defense']['team']:<20} "
              f"{stats['best_win_streak']['team']} ({stats['best_win_streak']['games']}V)")
    except Exception as e:
        print(f"{s:<12} ERREUR : {e}")


In [ ]:
import plotly.graph_objects as go

COLORS = ['#00C46A', '#2FD9E0', '#FFB100', '#FF3B5C', '#9B59B6', '#E67E22']

LOGOS = {
    "Arsenal":       "https://resources.premierleague.com/premierleague/badges/100/t3.png",
    "Aston Villa":   "https://resources.premierleague.com/premierleague/badges/100/t7.png",
    "Chelsea":       "https://resources.premierleague.com/premierleague/badges/100/t8.png",
    "Liverpool":     "https://resources.premierleague.com/premierleague/badges/100/t14.png",
    "Man City":      "https://resources.premierleague.com/premierleague/badges/100/t43.png",
    "Man United":    "https://resources.premierleague.com/premierleague/badges/100/t1.png",
    "Newcastle":     "https://resources.premierleague.com/premierleague/badges/100/t4.png",
    "Tottenham":     "https://resources.premierleague.com/premierleague/badges/100/t6.png",
    "Brighton":      "https://resources.premierleague.com/premierleague/badges/100/t36.png",
    "Leicester":     "https://resources.premierleague.com/premierleague/badges/100/t13.png",
    "Bournemouth":   "https://resources.premierleague.com/premierleague/badges/100/t91.png",
    "Nottm Forest":  "https://resources.premierleague.com/premierleague/badges/100/t17.png",
    "Nott'm Forest": "https://resources.premierleague.com/premierleague/badges/100/t17.png",
    "West Ham":      "https://resources.premierleague.com/premierleague/badges/100/t21.png",
    "Everton":       "https://resources.premierleague.com/premierleague/badges/100/t11.png",
    "Fulham":        "https://resources.premierleague.com/premierleague/badges/100/t54.png",
    "Wolves":        "https://resources.premierleague.com/premierleague/badges/100/t39.png",
    "Brentford":     "https://resources.premierleague.com/premierleague/badges/100/t94.png",
    "Leeds":         "https://resources.premierleague.com/premierleague/badges/100/t2.png",
    "Leeds United":  "https://resources.premierleague.com/premierleague/badges/100/t2.png",
    "Sunderland":    "https://resources.premierleague.com/premierleague/badges/100/t56.png",
    "Newcastle United": "https://resources.premierleague.com/premierleague/badges/100/t4.png",
}
FALLBACK = "https://resources.premierleague.com/premierleague/photos/players/110x140/Photo-Missing.png"

def build_animated_top6(evo_df, season_display):
    """
    Graphique Plotly animé avec frames natives (play/pause intégré).
    Annotations écusson + nom en bout de ligne sur la frame finale.
    """
    teams      = evo_df["team"].unique().tolist()
    matchweeks = sorted(evo_df["matchweek"].unique())
    max_mw     = matchweeks[-1]
    max_pts    = evo_df["pts"].max() + 8

    # ── Traces de base (frame finale complète) ────────────────────────────────
    traces = []
    for i, team in enumerate(teams):
        td = evo_df[evo_df["team"] == team]
        traces.append(go.Scatter(
            x=td["matchweek"].tolist(),
            y=td["pts"].tolist(),
            name=team,
            mode="lines+markers",
            line=dict(color=COLORS[i % len(COLORS)], width=2.5),
            marker=dict(
                size=5,
                color=COLORS[i % len(COLORS)],
            ),
            hovertemplate=f"<b>{team}</b><br>J%{{x}} — %{{y}} pts<extra></extra>",
        ))

    # ── Annotations finales : écusson + nom en bout de courbe ────────────────
    final_annotations = []
    final_images      = []

    for i, team in enumerate(teams):
        last_row = evo_df[(evo_df["team"] == team) & (evo_df["matchweek"] == max_mw)]
        if last_row.empty:
            continue
        last_pts = last_row["pts"].iloc[0]
        logo     = LOGOS.get(team, FALLBACK)

        # Image écusson
        final_images.append(dict(
            source  = logo,
            xref    = "x", yref = "y",
            x       = max_mw + 0.3,
            y       = last_pts,
            sizex   = 1.8,
            sizey   = 4.5,
            xanchor = "left",
            yanchor = "middle",
            layer   = "above",
        ))

        # Texte nom
        final_annotations.append(dict(
            x         = max_mw + 2.3,
            y         = last_pts,
            xref      = "x", yref = "y",
            text      = f"<b>{team}</b>",
            showarrow = False,
            font      = dict(color=COLORS[i % len(COLORS)], size=11,
                             family="IBM Plex Sans"),
            xanchor   = "left",
            yanchor   = "middle",
        ))

    # ── Frames pour l'animation ───────────────────────────────────────────────
    frames = []
    for mw in matchweeks:
        frame_traces = []
        frame_images = []
        frame_annots = []

        for i, team in enumerate(teams):
            td = evo_df[(evo_df["team"] == team) & (evo_df["matchweek"] <= mw)]
            frame_traces.append(go.Scatter(
                x=td["matchweek"].tolist(),
                y=td["pts"].tolist(),
            ))

            # Écusson en bout de courbe à chaque frame
            if not td.empty:
                last_pts = td["pts"].iloc[-1]
                logo     = LOGOS.get(team, FALLBACK)
                frame_images.append(dict(
                    source  = logo,
                    xref="x", yref="y",
                    x=mw + 0.3, y=last_pts,
                    sizex=1.8, sizey=4.5,
                    xanchor="left", yanchor="middle",
                    layer="above",
                ))
                frame_annots.append(dict(
                    x=mw + 2.3, y=last_pts,
                    xref="x", yref="y",
                    text=f"<b>{team}</b>",
                    showarrow=False,
                    font=dict(color=COLORS[i % len(COLORS)], size=11,
                              family="IBM Plex Sans"),
                    xanchor="left", yanchor="middle",
                ))

        frames.append(go.Frame(
            data   = frame_traces,
            name   = str(mw),
            layout = go.Layout(images=frame_images, annotations=frame_annots),
        ))

    # ── Layout ────────────────────────────────────────────────────────────────
    layout = go.Layout(
        plot_bgcolor  = "#0B0E14",
        paper_bgcolor = "#12182A",
        font          = dict(family="IBM Plex Mono", color="#E8E8E8", size=12),
        xaxis = dict(
            title     = "Journée",
            range     = [0, max_mw + 8],
            gridcolor = "rgba(139,146,168,0.1)",
            color     = "#8B92A8",
        ),
        yaxis = dict(
            title     = "Points cumulés",
            range     = [0, max_pts],
            gridcolor = "rgba(139,146,168,0.1)",
            color     = "#8B92A8",
        ),
        hovermode  = "x unified",
        showlegend = False,           # La légende est remplacée par les annotations
        height     = 520,
        margin     = dict(l=10, r=130, t=40, b=10),
        images      = final_images,
        annotations = final_annotations,

        # Contrôles Play/Pause natifs Plotly
        updatemenus = [dict(
            type       = "buttons",
            showactive = False,
            x=0.0, y=1.12, xanchor="left", yanchor="top",
            buttons = [
                dict(
                    label  = "Lancer",
                    method = "animate",
                    args   = [None, dict(
                        frame           = dict(duration=80, redraw=True),
                        fromcurrent     = True,
                        transition      = dict(duration=40, easing="linear"),
                        mode            = "immediate",
                    )],
                ),
                dict(
                    label  = "Pause",
                    method = "animate",
                    args   = [[None], dict(
                        frame       = dict(duration=0, redraw=False),
                        mode        = "immediate",
                        transition  = dict(duration=0),
                    )],
                ),
            ],
            bgcolor   = "#12182A",
            bordercolor = "#2FD9E0",
            font      = dict(color="#E8E8E8", family="IBM Plex Sans"),
        )],

        # Slider journée
        sliders = [dict(
            active     = len(matchweeks) - 1,
            currentvalue = dict(
                prefix   = "Journée ",
                font     = dict(color="#2FD9E0", family="IBM Plex Mono"),
                visible  = True,
                xanchor  = "center",
            ),
            pad        = dict(t=40),
            steps      = [
                dict(
                    method    = "animate",
                    label     = str(mw),
                    args      = [[str(mw)], dict(
                        mode            = "immediate",
                        frame           = dict(duration=0, redraw=True),
                        transition      = dict(duration=0),
                    )],
                )
                for mw in matchweeks
            ],
            bgcolor        = "#12182A",
            bordercolor    = "#8B92A8",
            tickcolor      = "#8B92A8",
            font           = dict(color="#8B92A8"),
        )],
    )

    fig = go.Figure(data=traces, layout=layout, frames=frames)
    return fig

# Test
evo  = top6_evolution(df_pl, '2025/26')
fig  = build_animated_top6(evo, '2025/26')
fig.show()


In [ ]:
title_counts = {}
for season in sorted(df_pl['Season_display'].unique()):
    s = season_standings(df_pl, season)
    if s.empty:
        continue
    champ = s.iloc[0]['team']
    title_counts[champ] = title_counts.get(champ, 0) + 1

titles_df = (pd.DataFrame(list(title_counts.items()), columns=['team', 'titles'])
               .sort_values('titles', ascending=False)
               .reset_index(drop=True))

print(titles_df.to_string(index=False))


In [ ]:
import streamlit as st

# Simuler ce que Streamlit voit
html = """
<div style="background:#12182A;border:1px solid rgba(139,146,168,0.2);
            border-left:3px solid #00C46A;border-radius:6px;
            padding:0.9rem 1rem;">
    <div style="font-size:0.72rem;color:#8B92A8;">MEILLEURE ATTAQUE</div>
    <div>Man City</div>
    <div style="color:#00C46A;">77 buts · 2.03/match</div>
</div>
"""
print("HTML valide — longueur:", len(html))
print("unsafe_allow_html nécessaire : OUI")


---
## P200 — Classement saison en cours (à venir)

> Expérimenter ici avant d'intégrer dans la page P200.

In [ ]:
# Placeholder — sera développé lors de la session P200
print('Section P200 à développer')

---
## P300 — Prédictions matchs (à venir)

> Tester les features dynamiques et les prédictions du modèle stacking.

In [ ]:
# Placeholder — sera développé lors de la session P300
print('Section P300 à développer')

---
## P400 — Simulation Monte Carlo (à venir)

> Valider la logique de simulation avant de l'intégrer.

In [ ]:
# Placeholder — sera développé lors de la session P400
print('Section P400 à développer')

---
## P500 — Carte des stades (à venir)

In [ ]:
# Placeholder — sera développé lors de la session P500
print('Section P500 à développer')

In [ ]:
import joblib
import pandas as pd
import numpy as np
from pathlib import Path

# Charger le modèle stacking
models_dir = ROOT / 'models'

gb_model    = joblib.load(models_dir / 'stacking_gb_base.pkl')
rf_model    = joblib.load(models_dir / 'stacking_rf_base.pkl')
lr_model    = joblib.load(models_dir / 'stacking_lr_base.pkl')
scaler      = joblib.load(models_dir / 'stacking_scaler.pkl')
meta_model  = joblib.load(models_dir / 'stacking_meta_model.pkl')

print("Modeles charges")
print("Meta-model classes:", meta_model.classes_)

# Charger les fixtures 2026-27
fixtures = pd.read_csv(ROOT / 'data' / 'raw' / 'fixtures.csv')
print(f"\nFixtures shape : {fixtures.shape}")
print(f"Colonnes       : {fixtures.columns.tolist()}")
print(fixtures.head(5))


In [ ]:
# Voir toutes les clés disponibles dans les métadonnées
print("Clés disponibles :")
for k, v in metadata.items():
    print(f"  {k} : {v}")


In [ ]:
import json

# Charger les métadonnées du modèle
with open(models_dir / 'stacking_v3_metadata.json') as f:
    metadata = json.load(f)

print("Features attendues :")
for i, feat in enumerate(metadata['feature_cols']):
    print(f"  {i+1:>2}. {feat}")

print(f"\nTotal : {len(metadata['feature_cols'])} features")
print(f"\nAccuracy : {metadata.get('accuracy', 'N/A')}")


In [ ]:
feature_names = metadata['feature_names']

# Ce qu'on peut calculer depuis les matchs enregistrés en DB
calculable = [
    'Home_Form', 'Home_Rank', 'Home_Streak', 'Home_Goal_Diff',
    'Home_Rolling_GF', 'Home_Rolling_GA', 'Home_Points_Pace', 'Home_Match_Number',
    'Away_Form', 'Away_Rank', 'Away_Streak', 'Away_Goal_Diff',
    'Away_Rolling_GF', 'Away_Rolling_GA', 'Away_Points_Pace', 'Away_Match_Number',
    'Home_xG_Form', 'Away_xG_Form'
]

# Ce qu'on n'a pas pour 2026-27
manquant = [f for f in feature_names if f not in calculable]

print(f"Features calculables depuis DB  : {len(calculable)}")
print(f"Features manquantes             : {len(manquant)}")
print("\nManquantes :")
for f in manquant:
    print(f"  - {f}")


In [ ]:
# Vérifier les fichiers externes disponibles
external_dir = ROOT / 'data' / 'external'

print("Fichiers disponibles dans data/external :")
for f in sorted(external_dir.glob('*.csv')):
    df_tmp = pd.read_csv(f, nrows=2)
    print(f"\n  {f.name}")
    print(f"  Colonnes : {df_tmp.columns.tolist()}")

# Vérifier aussi le fichier squad values
squad_file = external_dir / 'kaggle_squad_values_by_season.csv'
if squad_file.exists():
    sv = pd.read_csv(squad_file)
    print(f"\n\nSquad values shape : {sv.shape}")
    print(f"Saisons : {sorted(sv['season'].unique()) if 'season' in sv.columns else 'N/A'}")
    print(sv.head(3))


In [ ]:
epl = pd.read_csv(ROOT / 'data' / 'external' / 'epl_raw.csv')

print(f"Shape  : {epl.shape}")
print(f"Saisons: {sorted(epl['season'].unique())}")
print(f"Période: {epl['date'].min()} -> {epl['date'].max()}")

# Équipes présentes en 2025-26
last_season = epl['season'].max()
teams_epl = sorted(set(epl[epl['season'] == last_season]['home_team'].unique()) |
                   set(epl[epl['season'] == last_season]['away_team'].unique()))
print(f"\nÉquipes saison {last_season} ({len(teams_epl)}) :")
print(teams_epl)

# Équipes fixtures 2026-27
teams_fixtures = sorted(set(fixtures['home_team'].unique()) |
                        set(fixtures['away_team'].unique()))
print(f"\nÉquipes fixtures 2026-27 ({len(teams_fixtures)}) :")
print(teams_fixtures)

# Équipes manquantes dans epl_raw
missing = [t for t in teams_fixtures if t not in teams_epl]
print(f"\nÉquipes 2026-27 absentes de epl_raw : {missing}")


In [ ]:
# Mapping fixtures -> epl_raw
name_map = {
    'AFC Bournemouth':          'Bournemouth',
    'Brighton & Hove Albion':   'Brighton',
    'Leeds United':             'Leeds',
    'Tottenham Hotspur':        'Tottenham',
    'Coventry City':            None,   # promu, pas dans epl_raw
    'Hull City':                None,   # promu
    'Ipswich Town':             None,   # promu
}

# Calculer les stats moyennes par équipe sur la saison 2025/26
epl_2526 = epl[epl['season'] == 2526].copy()

team_stats_epl = {}
for team_fixture, team_epl in name_map.items():
    if team_epl is None:
        continue
    hm = epl_2526[epl_2526['home_team'] == team_epl]
    am = epl_2526[epl_2526['away_team'] == team_epl]
    team_stats_epl[team_fixture] = {
        'npxG' : round((hm['home_np_xg'].mean() + am['away_np_xg'].mean()) / 2, 3),
        'xPts' : round((hm['home_expected_points'].mean() + am['away_expected_points'].mean()) / 2, 3),
        'ppda' : round((hm['home_ppda'].mean() + am['away_ppda'].mean()) / 2, 3),
        'deep' : round((hm['home_deep_completions'].mean() + am['away_deep_completions'].mean()) / 2, 3),
    }

# Faire pareil pour les équipes directement présentes dans fixtures ET epl_raw
direct_teams = [t for t in teams_fixtures if t not in name_map]
for team in direct_teams:
    hm = epl_2526[epl_2526['home_team'] == team]
    am = epl_2526[epl_2526['away_team'] == team]
    team_stats_epl[team] = {
        'npxG' : round((hm['home_np_xg'].mean() + am['away_np_xg'].mean()) / 2, 3),
        'xPts' : round((hm['home_expected_points'].mean() + am['away_expected_points'].mean()) / 2, 3),
        'ppda' : round((hm['home_ppda'].mean() + am['away_ppda'].mean()) / 2, 3),
        'deep' : round((hm['home_deep_completions'].mean() + am['away_deep_completions'].mean()) / 2, 3),
    }

# Moyenne globale PL pour les promus
league_avg = {
    'npxG' : round(epl_2526[['home_np_xg', 'away_np_xg']].mean().mean(), 3),
    'xPts' : round(epl_2526[['home_expected_points', 'away_expected_points']].mean().mean(), 3),
    'ppda' : round(epl_2526[['home_ppda', 'away_ppda']].mean().mean(), 3),
    'deep' : round(epl_2526[['home_deep_completions', 'away_deep_completions']].mean().mean(), 3),
}
print("Moyenne PL (proxy promus) :", league_avg)

# Assigner la moyenne aux promus
for team in ['Coventry City', 'Hull City', 'Ipswich Town']:
    team_stats_epl[team] = league_avg.copy()

# Afficher le résultat
print(f"\nStats epl par équipe ({len(team_stats_epl)} équipes) :")
for team, s in sorted(team_stats_epl.items()):
    print(f"  {team:<30} npxG={s['npxG']}  xPts={s['xPts']}  ppda={s['ppda']}  deep={s['deep']}")


In [ ]:
sv = pd.read_csv(ROOT / 'data' / 'external' / 'kaggle_squad_values_by_season.csv')
tm = pd.read_csv(ROOT / 'data' / 'external' / 'transfermarkt_values_2324_2425.csv')

print("Squad values saisons disponibles:", sorted(sv['Season'].unique()))
print("Squad values colonnes:", sv.columns.tolist())
print()
print("Transfermarkt colonnes:", tm.columns.tolist())
print("Transfermarkt saisons:", sorted(tm['Season'].unique()))
print()

# Dernière saison dispo dans kaggle
last_sv_season = sv['Season'].max()
sv_last = sv[sv['Season'] == last_sv_season][['Team_Name','Squad_Value_Total','Squad_Value_Mean','Squad_Size']].copy()
print(f"Kaggle squad values saison {last_sv_season} ({len(sv_last)} équipes):")
print(sv_last.sort_values('Squad_Value_Total', ascending=False).to_string(index=False))


In [ ]:
# La bonne saison : 2526 (2025/26) comme proxy pour 2026-27
# ou 2627 si disponible
sv_2627 = sv[sv['Season'] == 2627][['Team_Name','Squad_Value_Total','Squad_Value_Mean','Squad_Size']].copy()
sv_2526 = sv[sv['Season'] == 2526][['Team_Name','Squad_Value_Total','Squad_Value_Mean','Squad_Size']].copy()

print(f"Saison 2627 : {len(sv_2627)} équipes")
print(sv_2627.sort_values('Squad_Value_Total', ascending=False).head(5).to_string(index=False))

print(f"\nSaison 2526 : {len(sv_2526)} équipes")
print(sv_2526.sort_values('Squad_Value_Total', ascending=False).head(5).to_string(index=False))

# Vérifier les noms d'équipes vs fixtures
print("\nNoms dans sv_2627:")
print(sorted(sv_2627['Team_Name'].unique()))


In [ ]:
# Mapping noms sv_2627 -> noms fixtures
sv_name_map = {
    'Arsenal FC':               'Arsenal',
    'Chelsea FC':               'Chelsea',
    'Liverpool FC':             'Liverpool',
    'Brentford FC':             'Brentford',
    'Everton FC':               'Everton',
    'Fulham FC':                'Fulham',
    'Brighton and Hove Albion': 'Brighton & Hove Albion',
    'Sunderland AFC':           'Sunderland',
    # Les autres ont déjà le bon nom
}

sv_2627['team'] = sv_2627['Team_Name'].replace(sv_name_map)
sv_2627 = sv_2627[['team', 'Squad_Value_Total', 'Squad_Value_Mean', 'Squad_Size']].copy()
sv_2627.columns = ['team', 'Home_Squad_Value', 'Home_Squad_Value_Mean', 'Home_Squad_Size']

# Vérifier la couverture
missing_sv = [t for t in teams_fixtures if t not in sv_2627['team'].values]
print(f"Équipes sans squad value : {missing_sv}")

# Construire la table complète des features statiques par équipe
static_features = sv_2627.set_index('team').copy()

# Ajouter les features epl_raw
for team in teams_fixtures:
    epl_s = team_stats_epl.get(team, league_avg)
    static_features.loc[team, 'Home_npxG']   = epl_s['npxG']
    static_features.loc[team, 'Away_npxG']   = epl_s['npxG']
    static_features.loc[team, 'Home_xPoints'] = epl_s['xPts']
    static_features.loc[team, 'Away_xPoints'] = epl_s['xPts']
    static_features.loc[team, 'Home_PPDA']   = epl_s['ppda']
    static_features.loc[team, 'Away_PPDA']   = epl_s['ppda']
    static_features.loc[team, 'Home_Deep']   = epl_s['deep']
    static_features.loc[team, 'Away_Deep']   = epl_s['deep']

# Dupliquer les colonnes squad value pour Away
static_features['Away_Squad_Value']      = static_features['Home_Squad_Value']
static_features['Away_Squad_Value_Mean'] = static_features['Home_Squad_Value_Mean']
static_features['Away_Squad_Size']       = static_features['Home_Squad_Size']

print(f"\nStatic features shape : {static_features.shape}")
print(static_features[['Home_Squad_Value', 'Home_npxG', 'Home_PPDA']].sort_values('Home_Squad_Value', ascending=False))



In [ ]:
def predict_match(home_team, away_team, match_number,
                  dynamic_stats, static_features, feature_names):
    """
    Prédit le résultat d'un match avec le modèle stacking.

    dynamic_stats : dict {team: {Form, Rank, Streak, Goal_Diff,
                                  Rolling_GF, Rolling_GA, Points_Pace,
                                  xG_Form}}
    static_features : DataFrame indexé par équipe
    """
    h = dynamic_stats[home_team]
    a = dynamic_stats[away_team]
    hs = static_features.loc[home_team]
    as_ = static_features.loc[away_team]

    row = {
        'Home_Form'            : h['Form'],
        'Home_Rank'            : h['Rank'],
        'Home_Streak'          : h['Streak'],
        'Home_Goal_Diff'       : h['Goal_Diff'],
        'Home_Rolling_GF'      : h['Rolling_GF'],
        'Home_Rolling_GA'      : h['Rolling_GA'],
        'Home_Points_Pace'     : h['Points_Pace'],
        'Home_Match_Number'    : match_number,
        'Away_Form'            : a['Form'],
        'Away_Rank'            : a['Rank'],
        'Away_Streak'          : a['Streak'],
        'Away_Goal_Diff'       : a['Goal_Diff'],
        'Away_Rolling_GF'      : a['Rolling_GF'],
        'Away_Rolling_GA'      : a['Rolling_GA'],
        'Away_Points_Pace'     : a['Points_Pace'],
        'Away_Match_Number'    : match_number,
        'Home_Squad_Value'     : hs['Home_Squad_Value'],
        'Home_Squad_Value_Mean': hs['Home_Squad_Value_Mean'],
        'Home_Squad_Size'      : hs['Home_Squad_Size'],
        'Away_Squad_Value'     : as_['Away_Squad_Value'],
        'Away_Squad_Value_Mean': as_['Away_Squad_Value_Mean'],
        'Away_Squad_Size'      : as_['Away_Squad_Size'],
        'Squad_Value_Delta'    : hs['Home_Squad_Value'] - as_['Away_Squad_Value'],
        'Home_npxG'            : hs['Home_npxG'],
        'Away_npxG'            : as_['Away_npxG'],
        'Home_xPoints'         : hs['Home_xPoints'],
        'Away_xPoints'         : as_['Away_xPoints'],
        'Home_PPDA'            : hs['Home_PPDA'],
        'Away_PPDA'            : as_['Away_PPDA'],
        'Home_Deep'            : hs['Home_Deep'],
        'Away_Deep'            : as_['Away_Deep'],
        'Home_xG_Form'         : h['xG_Form'],
        'Away_xG_Form'         : a['xG_Form'],
    }

    X = pd.DataFrame([row])[feature_names]

    # Stacking : prédictions des 3 modèles de base
    X_lr = scaler.transform(X)

    p_gb  = gb_model.predict_proba(X)
    p_rf  = rf_model.predict_proba(X)
    p_lr  = lr_model.predict_proba(X_lr)

    meta_X = np.hstack([p_gb, p_rf, p_lr])
    proba  = meta_model.predict_proba(meta_X)[0]
    pred   = meta_model.classes_[np.argmax(proba)]

    return {
        'pred'  : pred,
        'prob_A': round(proba[0], 4),
        'prob_D': round(proba[1], 4),
        'prob_H': round(proba[2], 4),
    }

# Test rapide : Arsenal vs Coventry (J1)
# Features dynamiques initiales (début de saison = tout à zéro/moyenne)
init_dynamic = {}
for team in teams_fixtures:
    init_dynamic[team] = {
        'Form'       : 0,
        'Rank'       : 10,
        'Streak'     : 0,
        'Goal_Diff'  : 0,
        'Rolling_GF' : 1.5,
        'Rolling_GA' : 1.5,
        'Points_Pace': 50,
        'xG_Form'    : static_features.loc[team, 'Home_npxG'],
    }

result = predict_match(
    'Arsenal', 'Coventry City',
    match_number=1,
    dynamic_stats=init_dynamic,
    static_features=static_features,
    feature_names=feature_names
)
print("Arsenal vs Coventry City :")
print(f"  Prédiction : {result['pred']}")
print(f"  P(H) = {result['prob_H']:.1%}  P(D) = {result['prob_D']:.1%}  P(A) = {result['prob_A']:.1%}")


In [ ]:
from scipy.stats import poisson

def estimate_poisson_lambdas(df_hist, home_team, away_team, season=None):
    """
    Estime les lambdas (buts attendus) pour un match via Poisson.
    Basé sur force d'attaque/défense de chaque équipe.
    """
    # Utiliser toutes les saisons ou une saison spécifique
    if season:
        d = df_hist[df_hist['Season_display'] == season]
    else:
        # 3 dernières saisons pour avoir assez de data
        recent = sorted(df_hist['Season_display'].unique())[-3:]
        d = df_hist[df_hist['Season_display'].isin(recent)]

    # Moyenne de buts par match dans le dataset
    avg_home_scored = d['FTHG'].mean()
    avg_away_scored = d['FTAG'].mean()

    # Force d'attaque et défense de chaque équipe
    def team_stats(team):
        hm = d[d['HomeTeam'] == team]
        am = d[d['AwayTeam'] == team]

        home_scored   = hm['FTHG'].mean() if len(hm) > 0 else avg_home_scored
        home_conceded = hm['FTAG'].mean() if len(hm) > 0 else avg_away_scored
        away_scored   = am['FTAG'].mean() if len(am) > 0 else avg_away_scored
        away_conceded = am['FTHG'].mean() if len(am) > 0 else avg_home_scored

        return {
            'att_home': home_scored   / avg_home_scored,
            'def_home': home_conceded / avg_away_scored,
            'att_away': away_scored   / avg_away_scored,
            'def_away': away_conceded / avg_home_scored,
        }

    h_stats = team_stats(home_team)
    a_stats = team_stats(away_team)

    # Lambda = moyenne PL * force attaque * faiblesse défense adverse
    lambda_home = avg_home_scored * h_stats['att_home'] * a_stats['def_away']
    lambda_away = avg_away_scored * a_stats['att_away'] * h_stats['def_home']

    return round(lambda_home, 3), round(lambda_away, 3)

def predict_score(home_team, away_team, df_hist, max_goals=6):
    """
    Prédit le score le plus probable et la distribution des scores.
    """
    # Mapping noms fixtures -> noms dans df_hist
    hist_name_map = {
        'AFC Bournemouth':          'Bournemouth',
        'Brighton & Hove Albion':   'Brighton',
        'Leeds United':             'Leeds',
        'Tottenham Hotspur':        'Tottenham',
        'Coventry City':            None,
        'Hull City':                None,
        'Ipswich Town':             'Ipswich',
        'Manchester City':          'Man City',
        'Manchester United':        'Man United',
        'Newcastle United':         'Newcastle',
        'Nottingham Forest':        "Nott'm Forest",
        'Sunderland':               'Sunderland',
    }

    h_hist = hist_name_map.get(home_team, home_team)
    a_hist = hist_name_map.get(away_team, away_team)

    # Promus sans historique PL : utiliser la moyenne
    if h_hist is None or a_hist is None:
        lambda_h, lambda_a = 1.5, 1.2
    else:
        lambda_h, lambda_a = estimate_poisson_lambdas(df_hist, h_hist, a_hist)

    # Matrice de probabilités des scores
    score_matrix = np.zeros((max_goals + 1, max_goals + 1))
    for i in range(max_goals + 1):
        for j in range(max_goals + 1):
            score_matrix[i, j] = poisson.pmf(i, lambda_h) * poisson.pmf(j, lambda_a)

    # Score le plus probable
    idx = np.unravel_index(score_matrix.argmax(), score_matrix.shape)
    most_likely_score = (idx[0], idx[1])

    # Top 5 scores les plus probables
    flat_idx = score_matrix.flatten().argsort()[::-1][:8]
    top_scores = [
        (divmod(i, max_goals + 1), round(score_matrix.flatten()[i] * 100, 1))
        for i in flat_idx
    ]

    # Probabilités globales
    prob_h = score_matrix[score_matrix > np.tril(score_matrix)].sum()
    prob_d = np.trace(score_matrix)
    prob_a = np.tril(score_matrix, -1).sum()

    # Correction
    prob_h = sum(score_matrix[i,j] for i in range(max_goals+1)
                                    for j in range(max_goals+1) if i > j)
    prob_d = sum(score_matrix[i,i] for i in range(max_goals+1))
    prob_a = sum(score_matrix[i,j] for i in range(max_goals+1)
                                    for j in range(max_goals+1) if j > i)

    return {
        'lambda_home'      : lambda_h,
        'lambda_away'      : lambda_a,
        'most_likely_score': most_likely_score,
        'top_scores'       : top_scores,
        'prob_H'           : round(prob_h, 3),
        'prob_D'           : round(prob_d, 3),
        'prob_A'           : round(prob_a, 3),
    }

# Test : Arsenal vs Coventry
result_score = predict_score('Arsenal', 'Coventry City', df_pl)

print(f"Arsenal vs Coventry City")
print(f"  Lambda home : {result_score['lambda_home']}  |  Lambda away : {result_score['lambda_away']}")
print(f"  Score le plus probable : {result_score['most_likely_score'][0]}-{result_score['most_likely_score'][1]}")
print(f"\n  P(H)={result_score['prob_H']:.1%}  P(D)={result_score['prob_D']:.1%}  P(A)={result_score['prob_A']:.1%}")
print(f"\n  Top 8 scores :")
for (h,a), prob in result_score['top_scores']:
    print(f"    {h}-{a}  :  {prob}%")


In [ ]:
from scipy.optimize import minimize

def find_lambdas_from_probas(prob_H, prob_D, prob_A, max_goals=10):
    """
    Trouve les lambdas Poisson cohérents avec les probabilités
    H/D/A prédites par le modèle stacking.
    """
    def poisson_probas(lambdas):
        lh, la = lambdas
        ph = pd = pa = 0
        for i in range(max_goals + 1):
            for j in range(max_goals + 1):
                p = poisson.pmf(i, lh) * poisson.pmf(j, la)
                if i > j: ph += p
                elif i == j: pd += p
                else: pa += p
        return ph, pd, pa

    def loss(lambdas):
        if lambdas[0] <= 0 or lambdas[1] <= 0:
            return 1e9
        ph, pd, pa = poisson_probas(lambdas)
        return (ph - prob_H)**2 + (pd - prob_D)**2 + (pa - prob_A)**2

    # Point de départ basé sur les probabilités
    lh0 = 1.5 * (prob_H / 0.45)
    la0 = 1.2 * (prob_A / 0.30)

    res = minimize(loss, [lh0, la0], method='Nelder-Mead',
                   options={'xatol': 1e-6, 'fatol': 1e-8, 'maxiter': 10000})

    return round(res.x[0], 3), round(res.x[1], 3)

def predict_score_from_model(home_team, away_team, prob_H, prob_D, prob_A, max_goals=6):
    """
    Score probable basé sur les probas du modèle stacking.
    """
    lambda_h, lambda_a = find_lambdas_from_probas(prob_H, prob_D, prob_A)

    score_matrix = np.zeros((max_goals + 1, max_goals + 1))
    for i in range(max_goals + 1):
        for j in range(max_goals + 1):
            score_matrix[i, j] = poisson.pmf(i, lambda_h) * poisson.pmf(j, lambda_a)

    idx = np.unravel_index(score_matrix.argmax(), score_matrix.shape)

    flat_idx = score_matrix.flatten().argsort()[::-1][:8]
    top_scores = [
        (divmod(i, max_goals + 1), round(score_matrix.flatten()[i] * 100, 1))
        for i in flat_idx
    ]

    return {
        'lambda_home'      : lambda_h,
        'lambda_away'      : lambda_a,
        'most_likely_score': (idx[0], idx[1]),
        'top_scores'       : top_scores,
    }

# Test avec les probas du stacking
stacking_result = predict_match(
    'Arsenal', 'Coventry City', 1,
    init_dynamic, static_features, feature_names
)

score_result = predict_score_from_model(
    'Arsenal', 'Coventry City',
    prob_H = stacking_result['prob_H'],
    prob_D = stacking_result['prob_D'],
    prob_A = stacking_result['prob_A'],
)

print(f"Arsenal vs Coventry City")
print(f"  Stacking  : P(H)={stacking_result['prob_H']:.1%}  P(D)={stacking_result['prob_D']:.1%}  P(A)={stacking_result['prob_A']:.1%}")
print(f"  Lambdas   : home={score_result['lambda_home']}  away={score_result['lambda_away']}")
print(f"  Score probable : {score_result['most_likely_score'][0]}-{score_result['most_likely_score'][1]}")
print(f"\n  Top 8 scores :")
for (h, a), prob in score_result['top_scores']:
    print(f"    {h}-{a}  :  {prob}%")


In [ ]:
# Simuler toute la journée 1 avec stacking + score Poisson
j1_fixtures = fixtures[fixtures['matchweek'] == 1].copy()

print(f"Journée 1 — {len(j1_fixtures)} matchs\n")
print(f"{'Match':<40} {'Pred':>5}  {'P(H)':>6} {'P(D)':>6} {'P(A)':>6}  {'Score':>5}")
print("-" * 75)

for _, row in j1_fixtures.iterrows():
    home = row['home_team']
    away = row['away_team']

    # Prédiction stacking
    pred = predict_match(
        home, away, 1,
        init_dynamic, static_features, feature_names
    )

    # Score probable
    score = predict_score_from_model(
        home, away,
        pred['prob_H'], pred['prob_D'], pred['prob_A']
    )

    match_str = f"{home} vs {away}"
    sc = score['most_likely_score']
    print(f"{match_str:<40} {pred['pred']:>5}  "
          f"{pred['prob_H']:>5.1%} {pred['prob_D']:>6.1%} {pred['prob_A']:>6.1%}  "
          f"{sc[0]}-{sc[1]:>1}")


In [ ]:
# Lire les résultats réels de la saison 2026-27 J1
txt_2627 = ROOT / 'data' / 'raw' / 'england-master' / '2026-27' / '1-premierleague.txt'
content_2627 = txt_2627.read_text(encoding='utf-8', errors='ignore')

# Parser les matchs de la journée 1
# Format : "  HH:MM   Home Team  X-X (X-X)  Away Team"
import re

match_pattern = re.compile(
    r'^\s+\d+:\d+\s+(.+?)\s+(\d+)-(\d+)\s+\(\d+-\d+\)\s+(.+?)\s*$',
    re.MULTILINE
)

# Extraire uniquement la Regular Season - 1
section = content_2627.split('▪ Regular Season - 2')[0]

real_results = []
for m in match_pattern.finditer(section):
    real_results.append({
        'home' : m.group(1).strip(),
        'score_h': int(m.group(2)),
        'score_a': int(m.group(3)),
        'away' : m.group(4).strip(),
        'result': 'H' if int(m.group(2)) > int(m.group(3)) else ('A' if int(m.group(2)) < int(m.group(3)) else 'D')
    })

print(f"Matchs parsés J1 : {len(real_results)}\n")
print(f"{'Match':<45} {'Réel':>5}  {'Score'}")
print("-" * 60)
for r in real_results:
    print(f"{r['home']} vs {r['away']:<30} {r['result']:>5}  {r['score_h']}-{r['score_a']}")


In [ ]:
# Afficher les 60 premières lignes du fichier pour voir le format exact
lines = content_2627.split('\n')
for i, line in enumerate(lines[:60]):
    print(f"{i:>3}  {repr(line)}")


In [ ]:
def parse_results_2627(content):
    """
    Parse le fichier 2026-27.
    Format : '    HH:MM  Home FC    v Away FC    X-X (X-X)'
    Retourne uniquement les matchs avec score.
    """
    # Pattern : ligne avec score X-X (X-X)
    match_pattern = re.compile(
        r'^\s+(?:\d+:\d+\s+)?(.+?)\s+v\s+(.+?)\s+(\d+)-(\d+)\s+\(\d+-\d+\)',
        re.MULTILINE
    )

    # Mapping noms england-master -> noms fixtures
    name_clean = {
        'Arsenal FC':               'Arsenal',
        'Coventry City FC':         'Coventry City',
        'Hull City AFC':            'Hull City',
        'Manchester United FC':     'Manchester United',
        'Ipswich Town FC':          'Ipswich Town',
        'Sunderland AFC':           'Sunderland',
        'Nottingham Forest FC':     'Nottingham Forest',
        'Leeds United FC':          'Leeds United',
        'Everton FC':               'Everton',
        'Crystal Palace FC':        'Crystal Palace',
        'Brentford FC':             'Brentford',
        'Tottenham Hotspur FC':     'Tottenham Hotspur',
        'Manchester City FC':       'Manchester City',
        'AFC Bournemouth':          'AFC Bournemouth',
        'Brighton & Hove Albion FC':'Brighton & Hove Albion',
        'Aston Villa FC':           'Aston Villa',
        'Newcastle United FC':      'Newcastle United',
        'Liverpool FC':             'Liverpool',
        'Fulham FC':                'Fulham',
        'Chelsea FC':               'Chelsea',
    }

    results = []
    current_matchday = None

    for line in content.split('\n'):
        # Détecter la journée
        md = re.match(r'^▪ Matchday (\d+)', line)
        if md:
            current_matchday = int(md.group(1))
            continue

        # Parser le match
        m = match_pattern.match(line)
        if m:
            home_raw = m.group(1).strip()
            away_raw = m.group(2).strip()
            score_h  = int(m.group(3))
            score_a  = int(m.group(4))

            home = name_clean.get(home_raw, home_raw)
            away = name_clean.get(away_raw, away_raw)
            result = 'H' if score_h > score_a else ('A' if score_h < score_a else 'D')

            results.append({
                'matchday': current_matchday,
                'home'    : home,
                'away'    : away,
                'score_h' : score_h,
                'score_a' : score_a,
                'result'  : result,
            })

    return pd.DataFrame(results)

real_df = parse_results_2627(content_2627)
print(f"Matchs joués : {len(real_df)}")
print(f"Journées     : {sorted(real_df['matchday'].unique())}")
print()
print(real_df.to_string(index=False))


In [ ]:
# Comparaison prédictions vs résultats réels J1
print(f"{'Match':<40} {'Réel':>4}  {'Pred':>4}  {'Score réel':>10}  {'Score prédit':>12}  {'OK?':>4}")
print("-" * 90)

correct_result  = 0
correct_score   = 0

for _, real in real_df.iterrows():
    home = real['home']
    away = real['away']

    # Skip si équipe pas dans nos features
    if home not in static_features.index or away not in static_features.index:
        print(f"  SKIP : {home} ou {away} absent des features")
        continue

    # Prédiction stacking
    pred = predict_match(
        home, away, 1,
        init_dynamic, static_features, feature_names
    )

    # Score probable
    score = predict_score_from_model(
        home, away,
        pred['prob_H'], pred['prob_D'], pred['prob_A']
    )

    sc = score['most_likely_score']
    real_score = f"{real['score_h']}-{real['score_a']}"
    pred_score = f"{sc[0]}-{sc[1]}"

    result_ok = '✓' if pred['pred'] == real['result'] else '✗'
    score_ok  = '✓' if sc[0] == real['score_h'] and sc[1] == real['score_a'] else ' '

    if pred['pred'] == real['result']:
        correct_result += 1
    if sc[0] == real['score_h'] and sc[1] == real['score_a']:
        correct_score += 1

    match_str = f"{home} vs {away}"
    print(f"{match_str:<40} {real['result']:>4}  {pred['pred']:>4}  "
          f"{real_score:>10}  {pred_score:>12}  {result_ok} {score_ok}")

print("-" * 90)
print(f"Résultats corrects : {correct_result}/10  ({correct_result/10:.0%})")
print(f"Scores corrects    : {correct_score}/10  ({correct_score/10:.0%})")


In [ ]:
# Précalculer les lambdas pour tous les matchs restants AVANT la simulation
from scipy.optimize import minimize

print("Précalcul des lambdas...")

lambda_cache = {}

for _, match in fixtures.iterrows():
    home = match['home_team']
    away = match['away_team']
    key  = (home, away)

    if key in lambda_cache:
        continue
    if home not in static_features.index or away not in static_features.index:
        lambda_cache[key] = (1.5, 1.2)
        continue

    # Prédiction avec features initiales (début de saison)
    pred = predict_match(home, away, 19,   # journée médiane
                         init_dynamic, static_features, feature_names)

    score = predict_score_from_model(
        home, away, pred['prob_H'], pred['prob_D'], pred['prob_A']
    )
    lambda_cache[key] = (score['lambda_home'], score['lambda_away'])

print(f"Lambdas précalculés : {len(lambda_cache)} paires de matchs")
print("Exemple Arsenal vs Coventry :", lambda_cache.get(('Arsenal', 'Coventry City')))


In [ ]:
import random
import time

def simulate_season_fast(fixtures, static_features, feature_names,
                         lambda_cache, played_results=None, n_simulations=10000):
    """
    Monte Carlo rapide — lambdas précalculés, pas de minimize dans la boucle.
    """
    teams = sorted(set(fixtures['home_team']) | set(fixtures['away_team']))

    def init_team_stats():
        return {t: {
            'points': 0, 'played': 0, 'gf': 0, 'ga': 0,
            'form_results': [],
            'Form': 0, 'Rank': 10, 'Streak': 0,
            'Goal_Diff': 0, 'Rolling_GF': 1.5, 'Rolling_GA': 1.5,
            'Points_Pace': 50,
            'xG_Form': float(static_features.loc[t, 'Home_npxG'])
                       if t in static_features.index else 1.5
        } for t in teams}

    def update_stats(ts, home, away, result, sh, sa):
        for team, is_home in [(home, True), (away, False)]:
            s = ts[team]
            s['played'] += 1
            gf = sh if is_home else sa
            ga = sa if is_home else sh
            s['gf'] += gf
            s['ga'] += ga
            s['Goal_Diff'] = s['gf'] - s['ga']

            if (is_home and result == 'H') or (not is_home and result == 'A'):
                s['points'] += 3
                s['form_results'].append(3)
                sv = 1
            elif result == 'D':
                s['points'] += 1
                s['form_results'].append(1)
                sv = 0
            else:
                s['form_results'].append(0)
                sv = -1

            s['form_results'] = s['form_results'][-5:]
            s['Form']         = sum(s['form_results'])
            s['Streak']       = (max(0, s['Streak']) + 1 if sv == 1
                                 else (min(0, s['Streak']) - 1 if sv == -1 else 0))
            if s['played'] > 0:
                s['Rolling_GF']  = s['gf'] / s['played']
                s['Rolling_GA']  = s['ga'] / s['played']
                s['Points_Pace'] = s['points'] / s['played'] * 38

    def update_ranks(ts):
        ranked = sorted(ts, key=lambda t: (ts[t]['points'],
                                           ts[t]['Goal_Diff'],
                                           ts[t]['gf']), reverse=True)
        for i, t in enumerate(ranked):
            ts[t]['Rank'] = i + 1

    def sim_match(home, away, mw, ts):
        """Tirer résultat + score depuis le cache de lambdas + probas stacking"""
        if home not in static_features.index or away not in static_features.index:
            r = random.random()
            result = 'H' if r < 0.45 else ('D' if r < 0.70 else 'A')
            sh, sa = (1, 0) if result == 'H' else ((0, 0) if result == 'D' else (0, 1))
            return result, sh, sa

        # Probas stacking avec features dynamiques actuelles
        dyn = {t: {
            'Form':        ts[t]['Form'],
            'Rank':        ts[t]['Rank'],
            'Streak':      ts[t]['Streak'],
            'Goal_Diff':   ts[t]['Goal_Diff'],
            'Rolling_GF':  ts[t]['Rolling_GF'],
            'Rolling_GA':  ts[t]['Rolling_GA'],
            'Points_Pace': ts[t]['Points_Pace'],
            'xG_Form':     ts[t]['xG_Form'],
        } for t in [home, away]}

        pred = predict_match(home, away, mw, dyn, static_features, feature_names)

        # Tirer résultat
        r = random.random()
        result = ('H' if r < pred['prob_H']
                  else ('D' if r < pred['prob_H'] + pred['prob_D'] else 'A'))

        # Tirer score depuis lambdas précalculés (pas de minimize)
        lh, la = lambda_cache.get((home, away), (1.5, 1.2))
        sh = int(np.random.poisson(lh))
        sa = int(np.random.poisson(la))

        # Cohérence score / résultat
        if result == 'H' and sh <= sa: sh, sa = sa + 1, sa
        elif result == 'A' and sa <= sh: sh, sa = sh, sh + 1
        elif result == 'D': sa = sh

        return result, sh, sa

    # Matchs joués / restants
    played_set = set()
    if played_results is not None:
        for _, r in played_results.iterrows():
            played_set.add((r['home_team'], r['away_team']))

    remaining = fixtures[~fixtures.apply(
        lambda r: (r['home_team'], r['away_team']) in played_set, axis=1
    )].copy()

    print(f"Matchs joués    : {len(played_set)}")
    print(f"Matchs restants : {len(remaining)}")
    print(f"Simulations     : {n_simulations:,}")

    # Compteurs
    top4_count  = {t: 0 for t in teams}
    title_count = {t: 0 for t in teams}
    releg_count = {t: 0 for t in teams}
    pts_sum     = {t: 0 for t in teams}

    t0 = time.time()

    for sim in range(n_simulations):

        # Progression tous les 100 sims
        if sim > 0 and sim % 100 == 0:
            elapsed = time.time() - t0
            eta     = elapsed / sim * (n_simulations - sim)
            print(f"  {sim}/{n_simulations} — {elapsed:.0f}s écoulées — ETA {eta:.0f}s")

        ts = init_team_stats()

        # Appliquer matchs joués
        if played_results is not None:
            for _, r in played_results.iterrows():
                update_stats(ts, r['home_team'], r['away_team'],
                             r['result'], r['score_h'], r['score_a'])
            update_ranks(ts)

        # Simuler matchs restants
        for mw in sorted(remaining['matchweek'].unique()):
            for _, match in remaining[remaining['matchweek'] == mw].iterrows():
                result, sh, sa = sim_match(
                    match['home_team'], match['away_team'], mw, ts
                )
                update_stats(ts, match['home_team'], match['away_team'],
                             result, sh, sa)
            update_ranks(ts)

        # Classement final
        final = sorted(teams,
                       key=lambda t: (ts[t]['points'], ts[t]['Goal_Diff'], ts[t]['gf']),
                       reverse=True)

        for i, t in enumerate(final):
            pos = i + 1
            pts_sum[t] += ts[t]['points']
            if pos == 1:  title_count[t] += 1
            if pos <= 4:  top4_count[t]  += 1
            if pos >= 18: releg_count[t] += 1

    elapsed = time.time() - t0
    print(f"\nTerminé en {elapsed:.1f}s ({elapsed/n_simulations*1000:.0f}ms/sim)")

    rows = []
    for t in teams:
        rows.append({
            'team'       : t,
            'prob_title' : round(title_count[t] / n_simulations * 100, 1),
            'prob_top4'  : round(top4_count[t]  / n_simulations * 100, 1),
            'prob_releg' : round(releg_count[t]  / n_simulations * 100, 1),
            'avg_points' : round(pts_sum[t]       / n_simulations, 1),
        })

    return (pd.DataFrame(rows)
              .sort_values('prob_top4', ascending=False)
              .reset_index(drop=True))

# Test avec 100 simulations d'abord pour mesurer la vitesse
played = real_df.rename(columns={'home': 'home_team', 'away': 'away_team'}).copy()

print("Test vitesse (100 sims)...")
sim_results = simulate_season_fast(
    fixtures, static_features, feature_names,
    lambda_cache=lambda_cache,
    played_results=played,
    n_simulations=100
)

print("\nTop 10 probabilités Top 4 :")
print(sim_results.head(10).to_string(index=False))


In [ ]:
def predict_batch(matches_df, team_stats, static_features, feature_names):
    """
    Prédit tous les matchs d'un DataFrame en un seul appel au modèle.
    matches_df : colonnes [home_team, away_team, matchweek]
    Retourne : DataFrame avec prob_H, prob_D, prob_A par ligne
    """
    rows = []
    for _, match in matches_df.iterrows():
        home = match['home_team']
        away = match['away_team']
        mw   = match['matchweek']

        if home not in static_features.index or away not in static_features.index:
            rows.append({'prob_H': 0.45, 'prob_D': 0.25, 'prob_A': 0.30})
            continue

        hs  = static_features.loc[home]
        as_ = static_features.loc[away]
        h   = team_stats[home]
        a   = team_stats[away]

        rows.append({
            'Home_Form'            : h['Form'],
            'Home_Rank'            : h['Rank'],
            'Home_Streak'          : h['Streak'],
            'Home_Goal_Diff'       : h['Goal_Diff'],
            'Home_Rolling_GF'      : h['Rolling_GF'],
            'Home_Rolling_GA'      : h['Rolling_GA'],
            'Home_Points_Pace'     : h['Points_Pace'],
            'Home_Match_Number'    : mw,
            'Away_Form'            : a['Form'],
            'Away_Rank'            : a['Rank'],
            'Away_Streak'          : a['Streak'],
            'Away_Goal_Diff'       : a['Goal_Diff'],
            'Away_Rolling_GF'      : a['Rolling_GF'],
            'Away_Rolling_GA'      : a['Rolling_GA'],
            'Away_Points_Pace'     : a['Points_Pace'],
            'Away_Match_Number'    : mw,
            'Home_Squad_Value'     : hs['Home_Squad_Value'],
            'Home_Squad_Value_Mean': hs['Home_Squad_Value_Mean'],
            'Home_Squad_Size'      : hs['Home_Squad_Size'],
            'Away_Squad_Value'     : as_['Away_Squad_Value'],
            'Away_Squad_Value_Mean': as_['Away_Squad_Value_Mean'],
            'Away_Squad_Size'      : as_['Away_Squad_Size'],
            'Squad_Value_Delta'    : hs['Home_Squad_Value'] - as_['Away_Squad_Value'],
            'Home_npxG'            : hs['Home_npxG'],
            'Away_npxG'            : as_['Away_npxG'],
            'Home_xPoints'         : hs['Home_xPoints'],
            'Away_xPoints'         : as_['Away_xPoints'],
            'Home_PPDA'            : hs['Home_PPDA'],
            'Away_PPDA'            : as_['Away_PPDA'],
            'Home_Deep'            : hs['Home_Deep'],
            'Away_Deep'            : as_['Away_Deep'],
            'Home_xG_Form'         : h['xG_Form'],
            'Away_xG_Form'         : a['xG_Form'],
        })

    X     = pd.DataFrame(rows)[feature_names]
    X_lr  = scaler.transform(X)
    p_gb  = gb_model.predict_proba(X)
    p_rf  = rf_model.predict_proba(X)
    p_lr  = lr_model.predict_proba(X_lr)
    meta  = meta_model.predict_proba(np.hstack([p_gb, p_rf, p_lr]))

    return meta  # shape (n_matches, 3) — colonnes A, D, H

def simulate_season_vectorized(fixtures, static_features, feature_names,
                                lambda_cache, played_results=None,
                                n_simulations=10000):
    """
    Monte Carlo vectorisé — une prédiction batch par journée au lieu d'une par match.
    """
    teams = sorted(set(fixtures['home_team']) | set(fixtures['away_team']))

    def init_team_stats():
        return {t: {
            'points': 0, 'played': 0, 'gf': 0, 'ga': 0,
            'form_results': [],
            'Form': 0, 'Rank': 10, 'Streak': 0,
            'Goal_Diff': 0, 'Rolling_GF': 1.5, 'Rolling_GA': 1.5,
            'Points_Pace': 50,
            'xG_Form': float(static_features.loc[t, 'Home_npxG'])
                       if t in static_features.index else 1.5
        } for t in teams}

    def update_stats(ts, home, away, result, sh, sa):
        for team, is_home in [(home, True), (away, False)]:
            s   = ts[team]
            s['played'] += 1
            gf  = sh if is_home else sa
            ga  = sa if is_home else sh
            s['gf'] += gf
            s['ga'] += ga
            s['Goal_Diff'] = s['gf'] - s['ga']

            if (is_home and result == 'H') or (not is_home and result == 'A'):
                s['points'] += 3; s['form_results'].append(3); sv = 1
            elif result == 'D':
                s['points'] += 1; s['form_results'].append(1); sv = 0
            else:
                s['form_results'].append(0); sv = -1

            s['form_results'] = s['form_results'][-5:]
            s['Form']         = sum(s['form_results'])
            s['Streak']       = (max(0, s['Streak']) + 1 if sv == 1
                                 else (min(0, s['Streak']) - 1 if sv == -1 else 0))
            if s['played'] > 0:
                s['Rolling_GF']  = s['gf'] / s['played']
                s['Rolling_GA']  = s['ga'] / s['played']
                s['Points_Pace'] = s['points'] / s['played'] * 38

    def update_ranks(ts):
        ranked = sorted(ts, key=lambda t: (ts[t]['points'],
                                           ts[t]['Goal_Diff'],
                                           ts[t]['gf']), reverse=True)
        for i, t in enumerate(ranked):
            ts[t]['Rank'] = i + 1

    # Matchs joués / restants
    played_set = set()
    if played_results is not None:
        for _, r in played_results.iterrows():
            played_set.add((r['home_team'], r['away_team']))

    remaining = fixtures[~fixtures.apply(
        lambda r: (r['home_team'], r['away_team']) in played_set, axis=1
    )].copy()

    matchweeks = sorted(remaining['matchweek'].unique())

    print(f"Matchs joués    : {len(played_set)}")
    print(f"Matchs restants : {len(remaining)}")
    print(f"Journées        : {len(matchweeks)}")
    print(f"Simulations     : {n_simulations:,}")

    # Compteurs
    top4_count  = {t: 0 for t in teams}
    title_count = {t: 0 for t in teams}
    releg_count = {t: 0 for t in teams}
    pts_sum     = {t: 0 for t in teams}

    t0 = time.time()

    for sim in range(n_simulations):

        if sim > 0 and sim % 200 == 0:
            elapsed = time.time() - t0
            eta     = elapsed / sim * (n_simulations - sim)
            print(f"  {sim}/{n_simulations} — {elapsed:.0f}s — ETA {eta:.0f}s")

        ts = init_team_stats()

        # Appliquer matchs joués
        if played_results is not None:
            for _, r in played_results.iterrows():
                update_stats(ts, r['home_team'], r['away_team'],
                             r['result'], r['score_h'], r['score_a'])
            update_ranks(ts)

        # Simuler journée par journée (batch)
        for mw in matchweeks:
            mw_df = remaining[remaining['matchweek'] == mw]

            # Prédiction batch pour toute la journée
            probas = predict_batch(mw_df, ts, static_features, feature_names)
            # probas colonnes : [A, D, H]

            for idx_row, (_, match) in enumerate(mw_df.iterrows()):
                home = match['home_team']
                away = match['away_team']
                pH   = probas[idx_row, 2]
                pD   = probas[idx_row, 1]

                # Tirer résultat
                r = random.random()
                result = 'H' if r < pH else ('D' if r < pH + pD else 'A')

                # Score depuis cache
                lh, la = lambda_cache.get((home, away), (1.5, 1.2))
                sh = int(np.random.poisson(lh))
                sa = int(np.random.poisson(la))
                if result == 'H' and sh <= sa: sh, sa = sa + 1, sa
                elif result == 'A' and sa <= sh: sh, sa = sh, sh + 1
                elif result == 'D': sa = sh

                update_stats(ts, home, away, result, sh, sa)

            update_ranks(ts)

        # Classement final
        final = sorted(teams,
                       key=lambda t: (ts[t]['points'], ts[t]['Goal_Diff'], ts[t]['gf']),
                       reverse=True)

        for i, t in enumerate(final):
            pos = i + 1
            pts_sum[t] += ts[t]['points']
            if pos == 1:  title_count[t] += 1
            if pos <= 4:  top4_count[t]  += 1
            if pos >= 18: releg_count[t] += 1

    elapsed = time.time() - t0
    print(f"\nTerminé en {elapsed:.1f}s  ({elapsed/n_simulations*1000:.0f}ms/sim)")

    rows = []
    for t in teams:
        rows.append({
            'team'       : t,
            'prob_title' : round(title_count[t] / n_simulations * 100, 1),
            'prob_top4'  : round(top4_count[t]  / n_simulations * 100, 1),
            'prob_releg' : round(releg_count[t]  / n_simulations * 100, 1),
            'avg_points' : round(pts_sum[t]       / n_simulations, 1),
        })

    return (pd.DataFrame(rows)
              .sort_values('prob_top4', ascending=False)
              .reset_index(drop=True))

# Test vitesse 100 sims
played = real_df.rename(columns={'home': 'home_team', 'away': 'away_team'}).copy()

print("Test vitesse (100 sims)...")
sim_results = simulate_season_vectorized(
    fixtures, static_features, feature_names,
    lambda_cache=lambda_cache,
    played_results=played,
    n_simulations=100
)

print("\nTop 10 :")
print(sim_results.head(10).to_string(index=False))


In [ ]:
# Précalculer les probas pour tous les 380 matchs UNE SEULE FOIS
# avec les features initiales (début de saison)
# La simulation utilise ces probas fixes — beaucoup plus rapide

print("Précalcul des probas pour tous les matchs...")

proba_cache = {}  # (home, away) -> (prob_H, prob_D, prob_A)

# Construire la matrice X pour tous les 380 matchs d'un coup
all_rows = []
all_keys = []

for _, match in fixtures.iterrows():
    home = match['home_team']
    away = match['away_team']
    mw   = match['matchweek']

    if home not in static_features.index or away not in static_features.index:
        proba_cache[(home, away)] = (0.45, 0.25, 0.30)
        continue

    hs  = static_features.loc[home]
    as_ = static_features.loc[away]

    all_rows.append({
        'Home_Form': 0, 'Home_Rank': 10, 'Home_Streak': 0,
        'Home_Goal_Diff': 0, 'Home_Rolling_GF': 1.5, 'Home_Rolling_GA': 1.5,
        'Home_Points_Pace': 50, 'Home_Match_Number': mw,
        'Away_Form': 0, 'Away_Rank': 10, 'Away_Streak': 0,
        'Away_Goal_Diff': 0, 'Away_Rolling_GF': 1.5, 'Away_Rolling_GA': 1.5,
        'Away_Points_Pace': 50, 'Away_Match_Number': mw,
        'Home_Squad_Value':      hs['Home_Squad_Value'],
        'Home_Squad_Value_Mean': hs['Home_Squad_Value_Mean'],
        'Home_Squad_Size':       hs['Home_Squad_Size'],
        'Away_Squad_Value':      as_['Away_Squad_Value'],
        'Away_Squad_Value_Mean': as_['Away_Squad_Value_Mean'],
        'Away_Squad_Size':       as_['Away_Squad_Size'],
        'Squad_Value_Delta':     hs['Home_Squad_Value'] - as_['Away_Squad_Value'],
        'Home_npxG':    hs['Home_npxG'],   'Away_npxG':    as_['Away_npxG'],
        'Home_xPoints': hs['Home_xPoints'],'Away_xPoints': as_['Away_xPoints'],
        'Home_PPDA':    hs['Home_PPDA'],   'Away_PPDA':    as_['Away_PPDA'],
        'Home_Deep':    hs['Home_Deep'],   'Away_Deep':    as_['Away_Deep'],
        'Home_xG_Form': hs['Home_npxG'],   'Away_xG_Form': as_['Away_npxG'],
    })
    all_keys.append((home, away))

# Un seul appel au modèle pour tous les 380 matchs
X_all    = pd.DataFrame(all_rows)[feature_names]
X_all_lr = scaler.transform(X_all)
p_gb_all = gb_model.predict_proba(X_all)
p_rf_all = rf_model.predict_proba(X_all)
p_lr_all = lr_model.predict_proba(X_all_lr)
meta_all = meta_model.predict_proba(np.hstack([p_gb_all, p_rf_all, p_lr_all]))

# Stocker dans le cache
for i, key in enumerate(all_keys):
    proba_cache[key] = (
        float(meta_all[i, 2]),  # prob_H
        float(meta_all[i, 1]),  # prob_D
        float(meta_all[i, 0]),  # prob_A
    )

print(f"Probas précalculées : {len(proba_cache)} matchs")
print(f"Arsenal vs Coventry : {proba_cache.get(('Arsenal','Coventry City'))}")
print(f"Man City vs Hull    : {proba_cache.get(('Manchester City','Hull City'))}")


In [ ]:
def simulate_season_cached(fixtures, proba_cache, lambda_cache,
                            played_results=None, n_simulations=10000):
    """
    Monte Carlo ultra-rapide — probas et lambdas précalculés.
    Zéro appel au modèle sklearn dans la boucle.
    """
    teams = sorted(set(fixtures['home_team']) | set(fixtures['away_team']))

    def init_team_stats():
        return {t: {'points': 0, 'played': 0, 'gf': 0, 'ga': 0,
                    'Goal_Diff': 0} for t in teams}

    def update_stats(ts, home, away, result, sh, sa):
        for team, is_home in [(home, True), (away, False)]:
            s  = ts[team]
            s['played'] += 1
            gf = sh if is_home else sa
            ga = sa if is_home else sh
            s['gf'] += gf
            s['ga'] += ga
            s['Goal_Diff'] = s['gf'] - s['ga']
            if (is_home and result == 'H') or (not is_home and result == 'A'):
                s['points'] += 3
            elif result == 'D':
                s['points'] += 1

    # Matchs joués / restants
    played_set = set()
    if played_results is not None:
        for _, r in played_results.iterrows():
            played_set.add((r['home_team'], r['away_team']))

    remaining = fixtures[~fixtures.apply(
        lambda r: (r['home_team'], r['away_team']) in played_set, axis=1
    )].reset_index(drop=True)

    # Pré-extraire les arrays numpy pour la boucle interne
    rem_home   = remaining['home_team'].values
    rem_away   = remaining['away_team'].values
    rem_mw     = remaining['matchweek'].values

    # Précalculer les arrays de probas et lambdas pour chaque match restant
    prob_H_arr = np.array([proba_cache.get((h, a), (0.45, 0.25, 0.30))[0]
                           for h, a in zip(rem_home, rem_away)])
    prob_D_arr = np.array([proba_cache.get((h, a), (0.45, 0.25, 0.30))[1]
                           for h, a in zip(rem_home, rem_away)])
    lh_arr     = np.array([lambda_cache.get((h, a), (1.5, 1.2))[0]
                           for h, a in zip(rem_home, rem_away)])
    la_arr     = np.array([lambda_cache.get((h, a), (1.5, 1.2))[1]
                           for h, a in zip(rem_home, rem_away)])

    n_rem = len(remaining)

    print(f"Matchs joués    : {len(played_set)}")
    print(f"Matchs restants : {n_rem}")
    print(f"Simulations     : {n_simulations:,}")

    # Compteurs
    top4_count  = {t: 0 for t in teams}
    title_count = {t: 0 for t in teams}
    releg_count = {t: 0 for t in teams}
    pts_sum     = {t: 0 for t in teams}

    t0 = time.time()

    for sim in range(n_simulations):
        if sim > 0 and sim % 1000 == 0:
            elapsed = time.time() - t0
            eta     = elapsed / sim * (n_simulations - sim)
            print(f"  {sim:>6}/{n_simulations} — {elapsed:.0f}s — ETA {eta:.0f}s")

        ts = init_team_stats()

        # Appliquer matchs joués
        if played_results is not None:
            for _, r in played_results.iterrows():
                update_stats(ts, r['home_team'], r['away_team'],
                             r['result'], r['score_h'], r['score_a'])

        # Tirer tous les résultats d'un coup (vectorisé numpy)
        rands   = np.random.random(n_rem)
        results = np.where(rands < prob_H_arr, 'H',
                  np.where(rands < prob_H_arr + prob_D_arr, 'D', 'A'))

        # Tirer tous les scores d'un coup
        sh_arr = np.random.poisson(lh_arr)
        sa_arr = np.random.poisson(la_arr)

        # Cohérence score/résultat
        for i in range(n_rem):
            r  = results[i]
            sh = int(sh_arr[i])
            sa = int(sa_arr[i])
            if r == 'H' and sh <= sa: sh = sa + 1
            elif r == 'A' and sa <= sh: sa = sh + 1
            elif r == 'D': sa = sh
            update_stats(ts, rem_home[i], rem_away[i], r, sh, sa)

        # Classement final
        final = sorted(teams,
                       key=lambda t: (ts[t]['points'],
                                      ts[t]['Goal_Diff'],
                                      ts[t]['gf']),
                       reverse=True)

        for i, t in enumerate(final):
            pos = i + 1
            pts_sum[t] += ts[t]['points']
            if pos == 1:  title_count[t] += 1
            if pos <= 4:  top4_count[t]  += 1
            if pos >= 18: releg_count[t] += 1

    elapsed = time.time() - t0
    print(f"\nTerminé en {elapsed:.1f}s  ({elapsed/n_simulations*1000:.1f}ms/sim)")

    rows = []
    for t in teams:
        rows.append({
            'team'       : t,
            'prob_title' : round(title_count[t] / n_simulations * 100, 1),
            'prob_top4'  : round(top4_count[t]  / n_simulations * 100, 1),
            'prob_releg' : round(releg_count[t]  / n_simulations * 100, 1),
            'avg_points' : round(pts_sum[t]       / n_simulations, 1),
        })

    return (pd.DataFrame(rows)
              .sort_values('prob_top4', ascending=False)
              .reset_index(drop=True))

# Test vitesse 1000 sims
played = real_df.rename(columns={'home': 'home_team', 'away': 'away_team'}).copy()

print("Test vitesse (1000 sims)...")
sim_results = simulate_season_cached(
    fixtures, proba_cache, lambda_cache,
    played_results=played,
    n_simulations=1000
)

print("\nRésultats :")
print(sim_results.to_string(index=False))


In [ ]:
print("Simulation 10,000...")
sim_results_10k = simulate_season_cached(
    fixtures, proba_cache, lambda_cache,
    played_results=played,
    n_simulations=10000
)

print("\nRésultats finaux (10,000 simulations) :")
print(sim_results_10k.to_string(index=False))


In [ ]:
import pickle

# Sauvegarder les caches précalculés
cache_dir = ROOT / 'app' / 'data'
cache_dir.mkdir(exist_ok=True)

with open(cache_dir / 'proba_cache.pkl', 'wb') as f:
    pickle.dump(proba_cache, f)

with open(cache_dir / 'lambda_cache.pkl', 'wb') as f:
    pickle.dump(lambda_cache, f)

print(f"proba_cache  : {len(proba_cache)} matchs sauvegardés")
print(f"lambda_cache : {len(lambda_cache)} matchs sauvegardés")
print(f"Destination  : {cache_dir}")


In [ ]:
# Saisons disponibles dans epl_raw
print("Saisons epl_raw:", sorted(epl['season'].unique()))

# Pour chaque saison cible, on a besoin de la saison précédente comme features
backtest_seasons = {
    '2021/22': {'target': 2122, 'features_from': 2021},
    '2022/23': {'target': 2223, 'features_from': 2122},
    '2023/24': {'target': 2324, 'features_from': 2223},
    '2024/25': {'target': 2425, 'features_from': 2324},
    '2025/26': {'target': 2526, 'features_from': 2425},
}

# Vérifier la couverture squad values pour chaque saison
sv = pd.read_csv(ROOT / 'data' / 'external' / 'kaggle_squad_values_by_season.csv')
print("\nSaisons squad values:", sorted(sv['Season'].unique()))

# Vérifier qu'on a bien les fixtures pour chaque saison target dans df_pl
print("\nSaisons dans df_pl:", sorted(df_pl['Season_display'].unique()))

# Test : équipes de la saison 2025/26 dans df_pl
teams_2526 = sorted(set(df_pl[df_pl['Season_display']=='2025/26']['HomeTeam'].dropna()) |
                    set(df_pl[df_pl['Season_display']=='2025/26']['AwayTeam'].dropna()))
print(f"\nÉquipes 2025/26 ({len(teams_2526)}):", teams_2526)


In [ ]:
# Mapping noms df_pl -> noms standardisés pour le modèle
PL_NAME_MAP = {
    'Bournemouth':       'AFC Bournemouth',
    'Brighton':          'Brighton & Hove Albion',
    'Leeds':             'Leeds United',
    'Tottenham':         'Tottenham Hotspur',
    'Man City':          'Manchester City',
    'Man United':        'Manchester United',
    'Newcastle':         'Newcastle United',
    "Nott'm Forest":     'Nottingham Forest',
    'Nottingham Forest': 'Nottingham Forest',
    'Wolves':            'Wolverhampton Wanderers',
    'West Ham':          'West Ham United',
    'Leicester':         'Leicester City',
}

# Mapping noms epl_raw -> noms standardisés
EPL_RAW_MAP = {
    'Bournemouth':       'AFC Bournemouth',
    'Brighton':          'Brighton & Hove Albion',
    'Leeds':             'Leeds United',
    'Tottenham':         'Tottenham Hotspur',
    'Man City':          'Manchester City',
    'Man United':        'Manchester United',
    'Newcastle United':  'Newcastle United',
    "Nott'm Forest":     'Nottingham Forest',
    'Wolves':            'Wolverhampton Wanderers',
    'West Ham':          'West Ham United',
    'Leicester':         'Leicester City',
}

def build_backtest_static_features(season_features, season_target):
    """
    Construire les features statiques pour une saison cible
    en utilisant les données de la saison précédente.

    season_features : int (ex: 2425) — saison source des features
    season_target   : int (ex: 2526) — saison à simuler
    """
    # Équipes de la saison cible
    target_df    = df_pl[df_pl['Season_display'] ==
                         next(k for k,v in backtest_seasons.items()
                              if v['target'] == season_target)]
    teams_raw    = sorted(set(target_df['HomeTeam'].dropna()) |
                          set(target_df['AwayTeam'].dropna()))
    teams_std    = [PL_NAME_MAP.get(t, t) for t in teams_raw]

    # Squad values saison target (si dispo) ou saison précédente
    sv_season = season_target if season_target in sv['Season'].values else season_features
    sv_data   = sv[sv['Season'] == sv_season][
        ['Team_Name', 'Squad_Value_Total', 'Squad_Value_Mean', 'Squad_Size']
    ].copy()
    sv_data['team_std'] = sv_data['Team_Name'].apply(lambda x: EPL_RAW_MAP.get(x, x))
    sv_idx = sv_data.set_index('team_std')

    # Stats xG saison features
    epl_feat = epl[epl['season'] == season_features].copy()
    league_avg = {
        'npxG': float(epl_feat[['home_np_xg','away_np_xg']].mean().mean()),
        'xPts': float(epl_feat[['home_expected_points','away_expected_points']].mean().mean()),
        'ppda': float(epl_feat[['home_ppda','away_ppda']].mean().mean()),
        'deep': float(epl_feat[['home_deep_completions','away_deep_completions']].mean().mean()),
    }

    rows = []
    for team_raw, team_std in zip(teams_raw, teams_std):
        # xG stats
        epl_name = EPL_RAW_MAP.get(team_raw, team_raw)
        hm = epl_feat[epl_feat['home_team'] == epl_name]
        am = epl_feat[epl_feat['away_team'] == epl_name]

        if len(hm) == 0 and len(am) == 0:
            ep = league_avg
        else:
            ep = {
                'npxG': float((hm['home_np_xg'].mean() + am['away_np_xg'].mean()) / 2),
                'xPts': float((hm['home_expected_points'].mean() + am['away_expected_points'].mean()) / 2),
                'ppda': float((hm['home_ppda'].mean() + am['away_ppda'].mean()) / 2),
                'deep': float((hm['home_deep_completions'].mean() + am['away_deep_completions'].mean()) / 2),
            }

        # Squad values
        sv_name = EPL_RAW_MAP.get(team_raw, team_raw)
        # Essayer plusieurs variantes de noms
        sv_row = None
        for name_try in [sv_name, team_raw, team_std]:
            if name_try in sv_idx.index:
                sv_row = sv_idx.loc[name_try]
                break

        sv_val  = float(sv_row['Squad_Value_Total']) if sv_row is not None else 3e8
        sv_mean = float(sv_row['Squad_Value_Mean'])  if sv_row is not None else 1e7
        sv_size = float(sv_row['Squad_Size'])         if sv_row is not None else 25

        rows.append({
            'team':                team_std,
            'team_raw':            team_raw,
            'Home_Squad_Value':    sv_val,
            'Home_Squad_Value_Mean': sv_mean,
            'Home_Squad_Size':     sv_size,
            'Away_Squad_Value':    sv_val,
            'Away_Squad_Value_Mean': sv_mean,
            'Away_Squad_Size':     sv_size,
            'Home_npxG':    ep['npxG'],  'Away_npxG':    ep['npxG'],
            'Home_xPoints': ep['xPts'],  'Away_xPoints': ep['xPts'],
            'Home_PPDA':    ep['ppda'],  'Away_PPDA':    ep['ppda'],
            'Home_Deep':    ep['deep'],  'Away_Deep':    ep['deep'],
        })

    return pd.DataFrame(rows).set_index('team')

# Test sur 2025/26 (features depuis 2024/25)
sf_2526 = build_backtest_static_features(
    season_features=2425,
    season_target=2526
)
print(f"Shape : {sf_2526.shape}")
print(sf_2526[['Home_Squad_Value','Home_npxG','Home_PPDA']].sort_values('Home_Squad_Value', ascending=False))


In [ ]:
# Débugger les noms dans squad values pour la saison 2526
sv_2526 = sv[sv['Season'] == 2526][['Team_Name','Squad_Value_Total']].copy()
print("Noms dans sv saison 2526 :")
print(sv_2526.sort_values('Squad_Value_Total', ascending=False).to_string(index=False))


In [ ]:
# Mapping complet noms squad values -> noms standardisés
SV_NAME_MAP_FULL = {
    'Arsenal FC':                'Arsenal',
    'Chelsea FC':                'Chelsea',
    'Liverpool FC':              'Liverpool',
    'Brentford FC':              'Brentford',
    'Everton FC':                'Everton',
    'Fulham FC':                 'Fulham',
    'Burnley FC':                'Burnley',
    'Sunderland AFC':            'Sunderland',
    'Brighton and Hove Albion':  'Brighton & Hove Albion',
    'AFC Bournemouth':           'AFC Bournemouth',
    'Manchester City':           'Manchester City',
    'Manchester United':         'Manchester United',
    'Tottenham Hotspur':         'Tottenham Hotspur',
    'Newcastle United':          'Newcastle United',
    'Nottingham Forest':         'Nottingham Forest',
    'Aston Villa':               'Aston Villa',
    'Crystal Palace':            'Crystal Palace',
    'West Ham United':           'West Ham United',
    'Leeds United':              'Leeds United',
    'Wolverhampton Wanderers':   'Wolverhampton Wanderers',
    # Saisons antérieures
    'Leicester City':            'Leicester City',
    'Watford FC':                'Watford',
    'Norwich City FC':           'Norwich City',
    'Sheffield United FC':       'Sheffield United',
    'Ipswich Town FC':           'Ipswich Town',
    'Luton Town FC':             'Luton Town',
    'Hull City AFC':             'Hull City',
    'Coventry City FC':          'Coventry City',
}

# Reconstruire l'index sv avec noms standardisés
sv_2526_fixed = sv[sv['Season'] == 2526].copy()
sv_2526_fixed['team_std'] = sv_2526_fixed['Team_Name'].map(SV_NAME_MAP_FULL)
sv_2526_fixed = sv_2526_fixed.dropna(subset=['team_std'])
sv_idx_fixed  = sv_2526_fixed.set_index('team_std')

print("Vérification Arsenal, Liverpool, Chelsea :")
for team in ['Arsenal', 'Liverpool', 'Chelsea', 'Brighton & Hove Albion']:
    if team in sv_idx_fixed.index:
        val = sv_idx_fixed.loc[team, 'Squad_Value_Total']
        print(f"  {team:<30} : {val/1e6:.0f}M")
    else:
        print(f"  {team:<30} : MANQUANT")


In [ ]:
def build_backtest_static_features_v2(season_features, season_target, season_display_target):
    """
    Version corrigée avec mapping SV complet.
    season_features      : int  (ex: 2425) — saison source des features xG
    season_target        : int  (ex: 2526) — saison squad values
    season_display_target: str  (ex: '2025/26') — saison dans df_pl
    """
    # Équipes de la saison cible depuis df_pl
    target_df = df_pl[df_pl['Season_display'] == season_display_target]
    teams_raw = sorted(set(target_df['HomeTeam'].dropna()) |
                       set(target_df['AwayTeam'].dropna()))

    # Normaliser les noms (df_pl -> standard)
    teams_std = [PL_NAME_MAP.get(t, t) for t in teams_raw]

    # Squad values avec mapping complet
    sv_season = sv[sv['Season'] == season_target].copy()
    sv_season['team_std'] = sv_season['Team_Name'].map(SV_NAME_MAP_FULL)
    sv_idx    = sv_season.dropna(subset=['team_std']).set_index('team_std')

    # Stats xG depuis epl_raw saison précédente
    epl_feat   = epl[epl['season'] == season_features].copy()
    league_avg = {
        'npxG': float(epl_feat[['home_np_xg','away_np_xg']].mean().mean()),
        'xPts': float(epl_feat[['home_expected_points','away_expected_points']].mean().mean()),
        'ppda': float(epl_feat[['home_ppda','away_ppda']].mean().mean()),
        'deep': float(epl_feat[['home_deep_completions','away_deep_completions']].mean().mean()),
    }

    rows = []
    for team_raw, team_std in zip(teams_raw, teams_std):
        # xG : chercher dans epl_raw avec nom normalisé
        epl_name = EPL_RAW_MAP.get(team_raw, team_raw)
        hm = epl_feat[epl_feat['home_team'] == epl_name]
        am = epl_feat[epl_feat['away_team'] == epl_name]
        ep = league_avg if (len(hm) == 0 and len(am) == 0) else {
            'npxG': float((hm['home_np_xg'].mean() + am['away_np_xg'].mean()) / 2),
            'xPts': float((hm['home_expected_points'].mean() + am['away_expected_points'].mean()) / 2),
            'ppda': float((hm['home_ppda'].mean() + am['away_ppda'].mean()) / 2),
            'deep': float((hm['home_deep_completions'].mean() + am['away_deep_completions'].mean()) / 2),
        }

        # Squad values
        sv_val  = float(sv_idx.loc[team_std, 'Squad_Value_Total']) if team_std in sv_idx.index else 3e8
        sv_mean = float(sv_idx.loc[team_std, 'Squad_Value_Mean'])  if team_std in sv_idx.index else 1e7
        sv_size = float(sv_idx.loc[team_std, 'Squad_Size'])         if team_std in sv_idx.index else 25

        rows.append({
            'team':                  team_std,
            'team_raw':              team_raw,
            'Home_Squad_Value':      sv_val,
            'Home_Squad_Value_Mean': sv_mean,
            'Home_Squad_Size':       sv_size,
            'Away_Squad_Value':      sv_val,
            'Away_Squad_Value_Mean': sv_mean,
            'Away_Squad_Size':       sv_size,
            'Home_npxG':    ep['npxG'],  'Away_npxG':    ep['npxG'],
            'Home_xPoints': ep['xPts'],  'Away_xPoints': ep['xPts'],
            'Home_PPDA':    ep['ppda'],  'Away_PPDA':    ep['ppda'],
            'Home_Deep':    ep['deep'],  'Away_Deep':    ep['deep'],
        })

    return pd.DataFrame(rows).set_index('team')

# Vérification
sf_v2 = build_backtest_static_features_v2(2425, 2526, '2025/26')
print("Squad values corrigées :")
print(sf_v2[['Home_Squad_Value']].sort_values('Home_Squad_Value', ascending=False))


In [ ]:
# Construire fixtures de la saison 2025/26 depuis df_pl
def build_fixtures_from_dfpl(season_display, name_map):
    """Extraire les fixtures d'une saison depuis df_pl avec noms standardisés"""
    s = df_pl[df_pl['Season_display'] == season_display][
        ['Date','HomeTeam','AwayTeam']
    ].copy().dropna()

    s['home_team']  = s['HomeTeam'].apply(lambda x: name_map.get(x, x))
    s['away_team']  = s['AwayTeam'].apply(lambda x: name_map.get(x, x))
    s['matchweek']  = 1  # on va recalculer

    # Assigner matchweek via assign_matchweek logic
    s = s.sort_values('Date').reset_index(drop=True)
    team_count = {}
    mws = []
    for _, row in s.iterrows():
        h = row['home_team']
        a = row['away_team']
        team_count[h] = team_count.get(h, 0) + 1
        team_count[a] = team_count.get(a, 0) + 1
        mws.append(max(team_count[h], team_count[a]))
    s['matchweek'] = mws

    return s[['home_team','away_team','matchweek']]

fixtures_2526 = build_fixtures_from_dfpl('2025/26', PL_NAME_MAP)
print(f"Fixtures 2025/26 : {len(fixtures_2526)} matchs")
print(f"Journées         : {fixtures_2526['matchweek'].nunique()}")
print(fixtures_2526.head(5).to_string(index=False))


In [ ]:
# Précalculer proba_cache et lambda_cache pour la saison 2025/26
from scipy.optimize import minimize
from scipy.stats import poisson

def build_backtest_caches(fixtures_df, static_features, feature_names):
    """Précalcul proba + lambda pour tous les matchs d'une saison"""
    teams = sorted(set(fixtures_df['home_team']) | set(fixtures_df['away_team']))

    init_dyn = {t: {
        'Form': 0, 'Rank': 10, 'Streak': 0, 'Goal_Diff': 0,
        'Rolling_GF': 1.5, 'Rolling_GA': 1.5, 'Points_Pace': 50,
        'xG_Form': float(static_features.loc[t, 'Home_npxG'])
                   if t in static_features.index else 1.5
    } for t in teams}

    all_rows, all_keys = [], []

    for _, match in fixtures_df.iterrows():
        home = match['home_team']
        away = match['away_team']
        mw   = match['matchweek']

        if home not in static_features.index or away not in static_features.index:
            continue

        hs  = static_features.loc[home]
        as_ = static_features.loc[away]
        h   = init_dyn[home]
        a   = init_dyn[away]

        all_rows.append({
            'Home_Form': h['Form'], 'Home_Rank': h['Rank'],
            'Home_Streak': h['Streak'], 'Home_Goal_Diff': h['Goal_Diff'],
            'Home_Rolling_GF': h['Rolling_GF'], 'Home_Rolling_GA': h['Rolling_GA'],
            'Home_Points_Pace': h['Points_Pace'], 'Home_Match_Number': mw,
            'Away_Form': a['Form'], 'Away_Rank': a['Rank'],
            'Away_Streak': a['Streak'], 'Away_Goal_Diff': a['Goal_Diff'],
            'Away_Rolling_GF': a['Rolling_GF'], 'Away_Rolling_GA': a['Rolling_GA'],
            'Away_Points_Pace': a['Points_Pace'], 'Away_Match_Number': mw,
            'Home_Squad_Value':      hs['Home_Squad_Value'],
            'Home_Squad_Value_Mean': hs['Home_Squad_Value_Mean'],
            'Home_Squad_Size':       hs['Home_Squad_Size'],
            'Away_Squad_Value':      as_['Away_Squad_Value'],
            'Away_Squad_Value_Mean': as_['Away_Squad_Value_Mean'],
            'Away_Squad_Size':       as_['Away_Squad_Size'],
            'Squad_Value_Delta':     hs['Home_Squad_Value'] - as_['Away_Squad_Value'],
            'Home_npxG': hs['Home_npxG'], 'Away_npxG': as_['Away_npxG'],
            'Home_xPoints': hs['Home_xPoints'], 'Away_xPoints': as_['Away_xPoints'],
            'Home_PPDA': hs['Home_PPDA'], 'Away_PPDA': as_['Away_PPDA'],
            'Home_Deep': hs['Home_Deep'], 'Away_Deep': as_['Away_Deep'],
            'Home_xG_Form': h['xG_Form'], 'Away_xG_Form': a['xG_Form'],
        })
        all_keys.append((home, away))

    # Un seul appel au modèle
    X      = pd.DataFrame(all_rows)[feature_names]
    X_lr   = scaler.transform(X)
    p_gb   = gb_model.predict_proba(X)
    p_rf   = rf_model.predict_proba(X)
    p_lr   = lr_model.predict_proba(X_lr)
    meta   = meta_model.predict_proba(np.hstack([p_gb, p_rf, p_lr]))

    proba_cache = {
        key: (float(meta[i,2]), float(meta[i,1]), float(meta[i,0]))
        for i, key in enumerate(all_keys)
    }

    # Lambda cache
    def find_lambdas(pH, pD, pA, max_g=10):
        def loss(lmb):
            if lmb[0] <= 0 or lmb[1] <= 0: return 1e9
            ph = pd_ = pa = 0.0
            for i in range(max_g+1):
                pi = poisson.pmf(i, lmb[0])
                for j in range(max_g+1):
                    p = pi * poisson.pmf(j, lmb[1])
                    if i > j: ph += p
                    elif i == j: pd_ += p
                    else: pa += p
            return (ph-pH)**2 + (pd_-pD)**2 + (pa-pA)**2
        res = minimize(loss, [1.5*pH/0.45, 1.2*pA/0.30], method='Nelder-Mead',
                      options={'xatol':1e-5,'fatol':1e-7,'maxiter':5000})
        return float(res.x[0]), float(res.x[1])

    lambda_cache = {
        key: find_lambdas(*proba)
        for key, proba in proba_cache.items()
    }

    return proba_cache, lambda_cache

# Construire les caches backtest 2025/26
print("Construction des caches backtest 2025/26...")
import time
t0 = time.time()
proba_bt, lambda_bt = build_backtest_caches(fixtures_2526, sf_v2, feature_names)
print(f"Terminé en {time.time()-t0:.1f}s")
print(f"proba  : {len(proba_bt)} matchs")
print(f"lambda : {len(lambda_bt)} matchs")
print(f"Exemple Arsenal vs Bournemouth : {proba_bt.get(('Arsenal','AFC Bournemouth'))}")


In [ ]:
# Sauvegarder les caches backtest 2025/26
import pickle

cache_dir = ROOT / 'app' / 'data'
cache_dir.mkdir(exist_ok=True)

with open(cache_dir / 'proba_cache_bt_2526.pkl', 'wb') as f:
    pickle.dump(proba_bt, f)

with open(cache_dir / 'lambda_cache_bt_2526.pkl', 'wb') as f:
    pickle.dump(lambda_bt, f)

print(f"Sauvegardes OK dans {cache_dir}")
print(f"proba_cache_bt_2526.pkl")
print(f"lambda_cache_bt_2526.pkl")


In [ ]:
# Simulation Monte Carlo backtest 2025/26
print("Simulation backtest 2025/26 (10,000 sims)...")
t0 = time.time()

sim_bt_2526 = simulate_season_cached(
    fixtures_2526,
    proba_cache=proba_bt,
    lambda_cache=lambda_bt,
    played_results=None,   # aucun match joué — simulation pré-saison complète
    n_simulations=10000
)

print(f"\nTerminé en {time.time()-t0:.1f}s")
print("\nClassement simulé 2025/26 :")
print(sim_bt_2526[['team','prob_title','prob_top4','prob_releg','avg_points']].to_string(index=False))


In [ ]:
# Fusionner simulation vs réel — colonne 'points' au lieu de 'pts'
comparison = sim_bt_2526.merge(
    real_2526[['team_std','pos','points','won','gf']].rename(columns={
        'team_std':'team','pos':'real_pos','points':'real_pts',
        'won':'real_won','gf':'real_gf'
    }),
    on='team', how='left'
)

comparison = comparison.sort_values('avg_points', ascending=False).reset_index(drop=True)
comparison.insert(0, 'sim_pos', range(1, len(comparison)+1))
comparison['pos_diff'] = comparison['sim_pos'] - comparison['real_pos']

print("=== Backtest 2025/26 : Simulation vs Réel ===\n")
print(f"{'Sim':>4}  {'Equipe':<25} {'Pts sim':>7}  {'Reel':>4}  {'Pts reel':>8}  {'Ecart':>6}")
print("-" * 65)
for _, r in comparison.iterrows():
    ecart = f"{r['pos_diff']:+.0f}" if not pd.isna(r['pos_diff']) else "N/A"
    arrow = "▲" if r['pos_diff'] < 0 else ("▼" if r['pos_diff'] > 0 else "=")
    rpos  = int(r['real_pos'])  if not pd.isna(r['real_pos'])  else "?"
    rpts  = int(r['real_pts'])  if not pd.isna(r['real_pts'])  else "?"
    print(f"{r['sim_pos']:>4}  {r['team']:<25} {r['avg_points']:>7.1f}  "
          f"{rpos:>4}  {rpts:>8}  {arrow} {ecart:>4}")

valid    = comparison.dropna(subset=['real_pos'])
mae      = abs(valid['pos_diff']).mean()
top4_sim  = set(comparison.head(4)['team'])
top4_real = set(real_2526.head(4)['team_std'])
top4_acc  = len(top4_sim & top4_real)

print(f"\nEcart moyen de position : {mae:.1f} places")
print(f"Top 4 simulé            : {top4_sim}")
print(f"Top 4 réel              : {top4_real}")
print(f"Correct                 : {top4_acc}/4  ({top4_acc/4:.0%})")


In [ ]:
# Sauvegarder les résultats backtest
import pickle

with open(cache_dir / 'backtest_2526_sim.pkl', 'wb') as f:
    pickle.dump(sim_bt_2526, f)

with open(cache_dir / 'backtest_2526_comparison.pkl', 'wb') as f:
    pickle.dump(comparison, f)

# Sauvegarder aussi les métriques
metrics_2526 = {
    'mae'           : round(mae, 1),
    'top4_acc'      : top4_acc,
    'top4_sim'      : list(top4_sim),
    'top4_real'     : list(top4_real),
    'season_display': '2025/26',
    'n_simulations' : 10000,
}

with open(cache_dir / 'backtest_2526_metrics.pkl', 'wb') as f:
    pickle.dump(metrics_2526, f)

print("Fichiers sauvegardés :")
import os
for f in os.listdir(cache_dir):
    size = os.path.getsize(cache_dir / f)
    print(f"  {f:<40} {size/1024:.1f} KB")
